# FreightQuote AI — Milestone 3 Upgrade

This notebook extends the existing FreightQuote app with six working, database-backed agents, a background worker, route recalculation, and storm-triggered weather avoidance.

**Implementation note:** The agents use transparent local rules and the existing SQLite data. They are executable agent modules, not calls to an external paid LLM API. Run the notebook from top to bottom in Google Colab. The worker starts with FastAPI and runs every 5 minutes by default (`AGENT_INTERVAL_SECONDS` can be set to another value, minimum 60 seconds).

New verification endpoints:
- `GET /api/agents/results` — latest persisted output from each agent
- `GET /api/agents/runs` — recent background/manual cycles
- `GET /api/routes/decisions` — route chosen for each shipment
- `POST /api/agents/weather/refresh` — manually refresh weather risk (Admin / Operations Manager)


**Dashboard UI:** After the server starts, open the `/milestone3` link shown below the server cell. Sign in with an existing app account; the demo Admin credentials are listed in the test cell.

In [25]:
!pip -q install fastapi "uvicorn[standard]" pyngrok bcrypt pyjwt python-multipart requests

In [26]:
import os, secrets
try:
    from google.colab import userdata
    def get_secret(n):
        try:
            return userdata.get(n)
        except Exception:
            return None
except ImportError:
    def get_secret(n):
        return os.environ.get(n)
for n in ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]:
    v = get_secret(n)
    if v:
        os.environ[n] = v
if not os.environ.get("JWT_SECRET"):
    os.environ["JWT_SECRET"] = secrets.token_hex(32)
    print("JWT_SECRET not in Colab Secrets -> using a temporary one for this session.")
if not (os.environ.get("EMAIL_ADDRESS") and os.environ.get("EMAIL_PASSWORD")):
    print("NOTE: email not configured - reset codes will be printed in this output instead of emailed.")
os.makedirs("static", exist_ok=True)
print("Ready (secret values are never printed).")

Ready (secret values are never printed).


In [27]:
%%writefile db.py
"""db.py - SQLite helpers + one shared schema and demo seed data for all five members."""
import os
import random
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone

DB_PATH = os.environ.get("DB_PATH", "freightquote_full.db")

SCHEMA = """
CREATE TABLE IF NOT EXISTS users (
    id               INTEGER PRIMARY KEY AUTOINCREMENT,
    email            TEXT NOT NULL UNIQUE COLLATE NOCASE,
    username         TEXT NOT NULL UNIQUE COLLATE NOCASE,
    password_hash    TEXT NOT NULL,
    role             TEXT NOT NULL DEFAULT 'Customer',
    locked           INTEGER NOT NULL DEFAULT 0,
    failed_attempts  INTEGER NOT NULL DEFAULT 0,
    customer_id      TEXT,
    created_at       TEXT NOT NULL,
    last_login       TEXT,
    reset_hash       TEXT,
    reset_expires    TEXT,
    reset_attempts   INTEGER NOT NULL DEFAULT 0
);
CREATE TABLE IF NOT EXISTS activity_log(
  id INTEGER PRIMARY KEY AUTOINCREMENT, actor TEXT, action TEXT, detail TEXT, ts TEXT);
CREATE TABLE IF NOT EXISTS carriers(
  id INTEGER PRIMARY KEY, name TEXT, cost_per_teu REAL, risk_score INTEGER, rank INTEGER, on_time_pct REAL);
CREATE TABLE IF NOT EXISTS ports(
  id INTEGER PRIMARY KEY, name TEXT, country TEXT, congestion_index INTEGER, congestion_level TEXT);
CREATE TABLE IF NOT EXISTS routes(
  id INTEGER PRIMARY KEY, origin TEXT, destination TEXT, via TEXT, cost_usd REAL, transit_days INTEGER);
CREATE TABLE IF NOT EXISTS carrier_rates(carrier TEXT, port TEXT, rate_usd REAL);
CREATE TABLE IF NOT EXISTS shipments(
  id TEXT PRIMARY KEY, customer_id TEXT, origin TEXT, destination TEXT, carrier TEXT,
  status TEXT, customs_status TEXT, weather_risk TEXT, eta TEXT);
CREATE TABLE IF NOT EXISTS quotes(
  id INTEGER PRIMARY KEY AUTOINCREMENT, customer_id TEXT, origin TEXT, destination TEXT, container TEXT DEFAULT '40ft',
  carrier TEXT, cost REAL, price REAL, margin_pct REAL, created_by TEXT, created_at TEXT);
CREATE TABLE IF NOT EXISTS storms(
  id INTEGER PRIMARY KEY, name TEXT, region TEXT, severity TEXT, wind_kts INTEGER);
CREATE TABLE IF NOT EXISTS agent_results(
  agent TEXT PRIMARY KEY, status TEXT, summary TEXT, payload TEXT, updated_at TEXT);
CREATE TABLE IF NOT EXISTS agent_runs(
  id INTEGER PRIMARY KEY AUTOINCREMENT, started_at TEXT, finished_at TEXT, status TEXT, summary TEXT);
CREATE TABLE IF NOT EXISTS route_decisions(
  shipment_id TEXT PRIMARY KEY, via TEXT, cost_usd REAL, transit_days INTEGER, reason TEXT, updated_at TEXT);
"""


@contextmanager
def db():
    conn = sqlite3.connect(DB_PATH, timeout=10)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def query(sql, params=(), one=False):
    with db() as c:
        rows = [dict(r) for r in c.execute(sql, params).fetchall()]
    return (rows[0] if rows else None) if one else rows


def execute(sql, params=()):
    with db() as c:
        return c.execute(sql, params).lastrowid


def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def next_customer_id():
    nums = []
    for r in query("SELECT customer_id FROM users WHERE customer_id LIKE 'C%'"):
        tail = (r["customer_id"] or "")[1:]
        if tail.isdigit():
            nums.append(int(tail))
    return f"C{max(nums, default=0) + 1:03d}"


DEMO_USERS = [  # email, username, password, role, customer_id
    ("admin@freightquote.ai", "admin", "Admin@123", "Admin", None),
    ("ops@freightquote.ai", "ops_manager", "Ops@12345", "Operations Manager", None),
    ("broker@freightquote.ai", "broker", "Broker@123", "Freight Broker", None),
    ("auditor@freightquote.ai", "auditor", "Audit@1234", "Auditor", None),
    ("customer@freightquote.ai", "customer_one", "Customer@123", "Customer", "C001"),
]


def init_db():
    with db() as c:
        c.executescript(SCHEMA)
    if query("SELECT COUNT(*) AS n FROM users", one=True)["n"] == 0:
        seed()


def seed():
    import security as sec
    rng = random.Random(42)
    t = now()
    today = datetime.now(timezone.utc)
    carriers = ["Maersk", "MSC", "CMA CGM", "Hapag-Lloyd", "COSCO", "Evergreen", "ONE", "ZIM"]
    ports = [("Rotterdam", "Netherlands", 82), ("Singapore", "Singapore", 74), ("Shanghai", "China", 91),
             ("Los Angeles", "USA", 88), ("Hamburg", "Germany", 55), ("Jebel Ali", "UAE", 48),
             ("Mumbai", "India", 63), ("Busan", "South Korea", 40), ("Felixstowe", "UK", 70)]
    routes = [("Shanghai", "Rotterdam", "Suez / Red Sea", 2450, 28), ("Singapore", "Hamburg", "Suez / Red Sea", 2300, 24),
              ("Mumbai", "Rotterdam", "Red Sea", 1900, 18), ("Busan", "Los Angeles", "North Pacific", 2100, 14),
              ("Shanghai", "Los Angeles", "North Pacific", 2250, 16), ("Jebel Ali", "Felixstowe", "Suez / Red Sea", 1750, 17),
              ("Singapore", "Mumbai", "Direct", 950, 7), ("Shanghai", "Hamburg", "Cape of Good Hope", 3100, 37)]
    with db() as c:
        for email, uname, pw, role, cust in DEMO_USERS:
            c.execute("INSERT INTO users(email,username,password_hash,role,customer_id,created_at) VALUES(?,?,?,?,?,?)",
                      (email, uname, sec.hash_password(pw), role, cust, t))
        for i, n in enumerate(carriers, 1):
            c.execute("INSERT INTO carriers VALUES(?,?,?,?,?,?)",
                      (i, n, rng.randint(1500, 2600), rng.randint(2, 9), i, round(rng.uniform(70, 96), 1)))
        for i, (n, ctry, idx) in enumerate(ports, 1):
            lvl = "High" if idx >= 80 else "Medium" if idx >= 60 else "Low"
            c.execute("INSERT INTO ports VALUES(?,?,?,?,?)", (i, n, ctry, idx, lvl))
        for i, r in enumerate(routes, 1):
            c.execute("INSERT INTO routes VALUES(?,?,?,?,?,?)", (i, *r))
        for carrier_name in carriers:
            base = rng.randint(1500, 2600)
            for pn, _, _ in ports:
                c.execute("INSERT INTO carrier_rates VALUES(?,?,?)", (carrier_name, pn, round(base * rng.uniform(0.9, 1.25))))
        statuses = ["In Transit", "In Transit", "In Transit", "Delayed", "Delayed", "Delivered", "Customs Hold"]
        for i in range(24):
            st = rng.choice(statuses)
            cs = "On Hold" if st == "Customs Hold" else "Cleared" if st == "Delivered" else rng.choice(["Cleared", "Pending"])
            o, d = rng.sample([p[0] for p in ports], 2)
            eta = (today + timedelta(days=rng.randint(-5, 30))).strftime("%Y-%m-%d")
            c.execute("INSERT INTO shipments VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{1001 + i}", f"C{rng.randint(1, 3):03d}", o, d, rng.choice(carriers), st, cs,
                       rng.choice(["None", "None", "Low", "High"]), eta))
        for i in range(18):
            o, d = rng.sample([p[0] for p in ports], 2)
            cost = rng.randint(1200, 3200)
            m = round(rng.uniform(4, 22), 1)
            price = round(cost / (1 - m / 100))
            c.execute("INSERT INTO quotes(customer_id,origin,destination,container,carrier,cost,price,margin_pct,created_by,created_at) "
                      "VALUES(?,?,?,?,?,?,?,?,?,?)",
                      (f"C{rng.randint(1, 3):03d}", o, d, "40ft", rng.choice(carriers), cost, price, m, "seed",
                       (today - timedelta(days=rng.randint(0, 40))).strftime("%Y-%m-%d")))
        for i, s in enumerate([("Storm Lena", "Red Sea", "High", 72), ("Storm Kai", "South China Sea", "Medium", 55),
                               ("Storm Omar", "Arabian Sea", "Medium", 48), ("Storm Vera", "North Atlantic", "Low", 35)], 1):
            c.execute("INSERT INTO storms VALUES(?,?,?,?,?)", (i, *s))

Overwriting db.py


In [28]:
%%writefile security.py
"""Validation, password hashing (bcrypt) and JWT helpers."""
import hashlib
import hmac
import os
import re
import secrets
from datetime import datetime, timedelta, timezone

import bcrypt
import jwt

JWT_SECRET = os.environ.get("JWT_SECRET", "")
if not JWT_SECRET:
    raise RuntimeError("JWT_SECRET is missing. Add it in Colab Secrets and run the secrets cell.")
JWT_ALG = "HS256"
JWT_HOURS = 8

EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9\-]+(\.[A-Za-z0-9\-]+)*\.[A-Za-z]{2,}$")
USERNAME_RE = re.compile(r"^[A-Za-z0-9_.\-]{3,30}$")


# ---------- validation ----------
def valid_email(email: str) -> bool:
    return bool(email) and len(email) <= 254 and bool(EMAIL_RE.match(email))


def valid_username(name: str) -> bool:
    return bool(USERNAME_RE.match(name or ""))


def password_problems(pw: str) -> list:
    """Return a list of unmet rules (empty list = password is fine)."""
    pw = pw or ""
    bad = []
    if len(pw) < 8:
        bad.append("at least 8 characters")
    if len(pw) > 64:
        bad.append("at most 64 characters")
    if not re.search(r"[A-Z]", pw):
        bad.append("an uppercase letter")
    if not re.search(r"[a-z]", pw):
        bad.append("a lowercase letter")
    if not re.search(r"\d", pw):
        bad.append("a digit")
    if not re.search(r"[^A-Za-z0-9]", pw):
        bad.append("a special character")
    return bad


# ---------- passwords ----------
def hash_password(pw: str) -> str:
    return bcrypt.hashpw(pw.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")


def verify_password(pw: str, hashed: str) -> bool:
    try:
        return bcrypt.checkpw(pw.encode("utf-8"), hashed.encode("utf-8"))
    except (ValueError, TypeError):
        return False


# Used so an unknown email takes as long to reject as a wrong password.
DUMMY_HASH = hash_password("dummy-Password-1!")


# ---------- JWT ----------
def make_token(user_id: int, email: str) -> str:
    now = datetime.now(timezone.utc)
    payload = {
        "sub": str(user_id),
        "email": email,
        "iat": now,
        "exp": now + timedelta(hours=JWT_HOURS),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALG)


def decode_token(token: str):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=[JWT_ALG])
    except jwt.PyJWTError:
        return None


# ---------- reset codes ----------
def new_reset_code() -> str:
    return f"{secrets.randbelow(10**6):06d}"


def hash_reset_code(code: str) -> str:
    return hmac.new(JWT_SECRET.encode(), code.encode(), hashlib.sha256).hexdigest()

Overwriting security.py


In [29]:
%%writefile mailer.py
"""Sends the password-reset code by email (Gmail SMTP, needs an App Password)."""
import os
import smtplib
from email.message import EmailMessage


def send_reset_code(to_email: str, code: str) -> bool:
    sender = os.environ.get("EMAIL_ADDRESS", "")
    password = os.environ.get("EMAIL_PASSWORD", "")
    if not sender or not password:
        return False
    msg = EmailMessage()
    msg["Subject"] = "FreightQuote AI - password reset code"
    msg["From"] = sender
    msg["To"] = to_email
    msg.set_content(
        f"Your FreightQuote AI password reset code is: {code}\n\n"
        "It expires in 15 minutes. If you did not request this, ignore this email."
    )
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=15) as s:
            s.login(sender, password)
            s.send_message(msg)
        return True
    except Exception as e:  # network / auth problems must not crash the API
        print(f"[mailer] could not send email: {e}")
        return False

Overwriting mailer.py


In [30]:
%%writefile auth.py
"""Member 1: authentication + profile routes, and user_of(request)."""
import sqlite3
from datetime import datetime, timedelta, timezone

from fastapi import APIRouter, HTTPException, Request, Response
from pydantic import BaseModel

import mailer
import security as sec
from db import db

router = APIRouter(prefix="/api/auth")

COOKIE = "fq_token"
MAX_ATTEMPTS = 5
RESET_MINUTES = 15
MAX_RESET_TRIES = 5
LOGIN_ERROR = "Invalid email or password."  # same text for unknown email and wrong password
LOCKED_MSG = "This account is locked. Please contact an administrator."


def _now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _clean_email(e: str) -> str:
    return (e or "").strip().lower()


def public_user(row) -> dict:
    """Safe view of a user row. Never includes the password hash or reset data."""
    return {
        "id": row["id"],
        "email": row["email"],
        "username": row["username"],
        "role": row["role"],
        "locked": bool(row["locked"]),
        "customer_id": row["customer_id"],
        "created_at": row["created_at"],
        "last_login": row["last_login"],
    }


def _fail(status: int, msg: str):
    raise HTTPException(status_code=status, detail=msg)


def _check_password_rules(pw: str, confirm: str):
    problems = sec.password_problems(pw)
    if problems:
        _fail(400, "Password must contain " + ", ".join(problems) + ".")
    if pw != confirm:
        _fail(400, "Passwords do not match.")


# ---------------------------------------------------------------- user_of
def user_of(request: Request) -> dict:
    """Read the JWT, confirm the user still exists and is not locked, return the real user.
    Called on every protected request, so a lock or role change applies immediately."""
    token = None
    header = request.headers.get("authorization", "")
    if header.lower().startswith("bearer "):
        token = header[7:].strip()
    token = token or request.cookies.get(COOKIE)
    if not token:
        _fail(401, "Please log in.")
    payload = sec.decode_token(token)
    if not payload:
        _fail(401, "Your session has expired. Please log in again.")
    try:
        uid = int(payload["sub"])
    except (KeyError, ValueError):
        _fail(401, "Invalid session.")
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE id = ?", (uid,)).fetchone()
    if not row:
        _fail(401, "Account no longer exists.")
    if row["locked"]:
        _fail(423, LOCKED_MSG)
    return public_user(row)


# ---------------------------------------------------------------- models
class SignupIn(BaseModel):
    email: str = ""
    username: str = ""
    password: str = ""
    confirm_password: str = ""


class LoginIn(BaseModel):
    email: str = ""
    password: str = ""


class ForgotIn(BaseModel):
    email: str = ""


class ResetIn(BaseModel):
    email: str = ""
    code: str = ""
    new_password: str = ""
    confirm_password: str = ""


class UsernameIn(BaseModel):
    username: str = ""


class ChangePasswordIn(BaseModel):
    current_password: str = ""
    new_password: str = ""
    confirm_password: str = ""


# ---------------------------------------------------------------- routes
@router.post("/signup", status_code=201)
def signup(body: SignupIn):
    email = _clean_email(body.email)
    username = (body.username or "").strip()
    if not sec.valid_email(email):
        _fail(400, "Please enter a valid email address.")
    if not sec.valid_username(username):
        _fail(400, "Username must be 3-30 characters: letters, numbers, dot, dash or underscore.")
    _check_password_rules(body.password, body.confirm_password)
    try:
        with db() as c:
            c.execute(
                "INSERT INTO users (email, username, password_hash, role, created_at) "
                "VALUES (?, ?, ?, 'Customer', ?)",
                (email, username, sec.hash_password(body.password), _now()),
            )
    except sqlite3.IntegrityError:
        _fail(409, "That email or username is already registered.")
    return {"message": "Account created. You can now log in."}


@router.post("/login")
def login(body: LoginIn, response: Response):
    email = _clean_email(body.email)
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE email = ?", (email,)).fetchone()
        if not row:
            sec.verify_password(body.password or "", sec.DUMMY_HASH)  # equalise timing
            _fail(401, LOGIN_ERROR)
        if row["locked"]:
            _fail(423, LOCKED_MSG)
        if not sec.verify_password(body.password or "", row["password_hash"]):
            attempts = row["failed_attempts"] + 1
            if attempts >= MAX_ATTEMPTS:
                c.execute("UPDATE users SET failed_attempts=?, locked=1 WHERE id=?", (attempts, row["id"]))
                c.commit()
                _fail(423, "Too many failed attempts. This account is now locked.")
            c.execute("UPDATE users SET failed_attempts=? WHERE id=?", (attempts, row["id"]))
            c.commit()
            _fail(401, LOGIN_ERROR)
        c.execute("UPDATE users SET failed_attempts=0, last_login=? WHERE id=?", (_now(), row["id"]))
        c.commit()
        row = c.execute("SELECT * FROM users WHERE id=?", (row["id"],)).fetchone()
    token = sec.make_token(row["id"], row["email"])
    response.set_cookie(
        COOKIE, token, httponly=True, samesite="lax", max_age=sec.JWT_HOURS * 3600, path="/"
    )
    return {"token": token, "user": public_user(row)}


@router.post("/logout")
def logout(response: Response):
    response.delete_cookie(COOKIE, path="/")
    return {"message": "Signed out."}


@router.get("/me")
def me(request: Request):
    return {"user": user_of(request)}


@router.post("/forgot")
def forgot(body: ForgotIn):
    """Always returns the same message so the endpoint cannot be used to find registered emails."""
    email = _clean_email(body.email)
    generic = {"message": "If that email is registered, a reset code has been sent."}
    if not sec.valid_email(email):
        _fail(400, "Please enter a valid email address.")
    with db() as c:
        row = c.execute("SELECT id FROM users WHERE email = ?", (email,)).fetchone()
        if not row:
            return generic
        code = sec.new_reset_code()
        expires = (datetime.now(timezone.utc) + timedelta(minutes=RESET_MINUTES)).isoformat(timespec="seconds")
        c.execute(
            "UPDATE users SET reset_hash=?, reset_expires=?, reset_attempts=0 WHERE id=?",
            (sec.hash_reset_code(code), expires, row["id"]),
        )
        c.commit()
    if not mailer.send_reset_code(email, code):
        # Colab-only fallback so you can still test when SMTP is not configured.
        print(f"[DEV] Email not sent. Reset code for {email}: {code}")
    return generic


@router.post("/reset")
def reset(body: ResetIn):
    email = _clean_email(body.email)
    bad = "Invalid or expired reset code."
    _check_password_rules(body.new_password, body.confirm_password)
    with db() as c:
        row = c.execute("SELECT * FROM users WHERE email = ?", (email,)).fetchone()
        if not row or not row["reset_hash"] or not row["reset_expires"]:
            _fail(400, bad)
        if row["reset_attempts"] >= MAX_RESET_TRIES:
            _fail(400, "Too many attempts. Please request a new code.")
        if datetime.fromisoformat(row["reset_expires"]) < datetime.now(timezone.utc):
            _fail(400, bad)
        import hmac
        if not hmac.compare_digest(row["reset_hash"], sec.hash_reset_code((body.code or "").strip())):
            c.execute("UPDATE users SET reset_attempts = reset_attempts + 1 WHERE id=?", (row["id"],))
            c.commit()
            _fail(400, bad)
        c.execute(
            "UPDATE users SET password_hash=?, reset_hash=NULL, reset_expires=NULL, "
            "reset_attempts=0, failed_attempts=0 WHERE id=?",
            (sec.hash_password(body.new_password), row["id"]),
        )
    return {"message": "Password updated. You can now log in."}


@router.put("/profile")
def update_profile(body: UsernameIn, request: Request):
    user = user_of(request)
    username = (body.username or "").strip()
    if not sec.valid_username(username):
        _fail(400, "Username must be 3-30 characters: letters, numbers, dot, dash or underscore.")
    try:
        with db() as c:
            c.execute("UPDATE users SET username=? WHERE id=?", (username, user["id"]))
    except sqlite3.IntegrityError:
        _fail(409, "That username is already taken.")
    return {"message": "Username updated.", "user": user_of(request)}


@router.post("/password")
def change_password(body: ChangePasswordIn, request: Request):
    user = user_of(request)
    with db() as c:
        row = c.execute("SELECT password_hash FROM users WHERE id=?", (user["id"],)).fetchone()
        if not sec.verify_password(body.current_password or "", row["password_hash"]):
            _fail(400, "Current password is incorrect.")
        _check_password_rules(body.new_password, body.confirm_password)
        if body.new_password == body.current_password:
            _fail(400, "New password must be different from the current password.")
        c.execute(
            "UPDATE users SET password_hash=? WHERE id=?",
            (sec.hash_password(body.new_password), user["id"]),
        )
    return {"message": "Password changed."}

Overwriting auth.py


In [31]:
%%writefile rbac.py
"""Member 2: five roles, permission matrix and server-side access control."""
from fastapi import HTTPException, Request

import auth
from db import db, next_customer_id

ROLES = ["Admin", "Operations Manager", "Freight Broker", "Auditor", "Customer"]

# page id -> label shown in the menu ("admin" is the Users page that Member 3 builds)
PAGE_LABELS = {
    "copilot": "AI Copilot",
    "route": "Route AI",
    "quotes": "Spot Quotes",
    "carriers": "Carriers",
    "weather": "Weather Risk",
    "margin": "Margin",
    "customs": "Customs",
    "admin": "Users",
}

# Which pages each role may open (matches the permission matrix in the task document)
ROLE_MENU = {
    "Admin": ["copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "admin"],
    "Operations Manager": ["copilot", "route", "carriers", "weather", "customs"],
    "Freight Broker": ["copilot", "quotes", "carriers", "margin"],
    "Auditor": ["copilot", "route", "quotes", "carriers", "weather", "margin", "customs"],
    "Customer": ["copilot", "quotes"],
}

ALL_CAPS = [
    "copilot.use",
    "route.view", "route.reroute",
    "quotes.view", "quotes.create",
    "carriers.view",
    "weather.view", "weather.inject_storm",
    "margin.view",
    "customs.view", "customs.edit",
    "agents.run",
    "users.manage",
]

# What each role may do (capability names)
ROLE_CAPS = {
    "Admin": list(ALL_CAPS),
    "Operations Manager": [
        "copilot.use", "route.view", "route.reroute", "carriers.view",
        "weather.view", "weather.inject_storm", "customs.view", "customs.edit", "agents.run",
    ],
    "Freight Broker": ["copilot.use", "quotes.view", "quotes.create", "carriers.view", "margin.view"],
    "Auditor": [  # read-only: only *.view capabilities
        "copilot.use", "route.view", "quotes.view", "carriers.view",
        "weather.view", "margin.view", "customs.view",
    ],
    "Customer": ["copilot.use", "quotes.view", "quotes.create"],  # own quotes only
}

# capability needed to open each page
PAGE_CAP = {
    "copilot": "copilot.use",
    "route": "route.view",
    "quotes": "quotes.view",
    "carriers": "carriers.view",
    "weather": "weather.view",
    "margin": "margin.view",
    "customs": "customs.view",
    "admin": "users.manage",
}


def menu_for(user: dict) -> list:
    """Only the pages this user's role may open."""
    return [{"id": p, "label": PAGE_LABELS[p]} for p in ROLE_MENU.get(user.get("role"), [])]


def caps_for(user: dict) -> list:
    return list(ROLE_CAPS.get(user.get("role"), []))


def has_cap(user: dict, capability: str) -> bool:
    return capability in ROLE_CAPS.get(user.get("role"), [])


def need(ctx: dict, capability: str):
    """Raise HTTP 403 when the user's role lacks the capability. ctx = the user from user_of()."""
    if not has_cap(ctx, capability):
        raise HTTPException(
            status_code=403,
            detail=f"Access denied (403): the {ctx.get('role')} role cannot use '{capability}'.",
        )


def need_page(ctx: dict, page_id: str):
    if page_id not in ROLE_MENU.get(ctx.get("role"), []):
        raise HTTPException(
            status_code=403,
            detail=f"Access denied (403): the {ctx.get('role')} role cannot open the '{page_id}' page.",
        )


def require(capability: str):
    """FastAPI dependency: log the user in (JWT, not locked), then check the capability."""
    def dep(request: Request) -> dict:
        user = auth.user_of(request)
        need(user, capability)
        return user
    return dep


def customer_scope(user: dict):
    """Customers: return their customer_id (assigned on first use, format C001). Other roles: None = see all."""
    if user.get("role") != "Customer":
        return None
    cid = user.get("customer_id")
    if cid:
        return cid
    new_id = next_customer_id()
    with db() as c:
        c.execute("UPDATE users SET customer_id=? WHERE id=? AND customer_id IS NULL", (new_id, user["id"]))
        row = c.execute("SELECT customer_id FROM users WHERE id=?", (user["id"],)).fetchone()
    return (row["customer_id"] if row else None) or new_id

Overwriting rbac.py


In [32]:
%%writefile analyst.py
"""analyst.py - Copilot intents. Every answer comes from a whitelisted, parameterised SQL query.
{CUST} is replaced by 'AND customer_id = ?' for Customers so they only see their own quotes."""

STOP = set("""the a an of is are was were be which what whats show me to in for all with has have do does there near any
please can you give list display tell on at by from and or it its i us we how information info related data details
available shipping generated created made current currently now so that this""".split())

SYNONYMS = {
    "lowest": "cheapest", "cheap": "cheapest", "inexpensive": "cheapest", "minimum": "cheapest", "least": "cheapest",
    "costs": "cost", "price": "cost", "prices": "cost", "pricing": "cost", "rate": "cost", "rates": "cost", "expensive": "cost",
    "late": "delayed", "delay": "delayed", "delays": "delayed", "overdue": "delayed",
    "congested": "congestion", "crowded": "congestion", "busy": "congestion", "jammed": "congestion",
    "held": "hold", "holds": "hold", "stuck": "hold", "blocked": "hold",
    "carriers": "carrier", "shipper": "carrier", "shippers": "carrier", "operator": "carrier", "operators": "carrier",
    "ports": "port", "harbor": "port", "harbour": "port", "harbors": "port",
    "routes": "route", "lanes": "route", "lane": "route", "paths": "route",
    "shipments": "shipment", "cargo": "shipment", "consignments": "shipment", "consignment": "shipment",
    "containers": "shipment", "orders": "shipment",
    "quotes": "quote", "quotations": "quote", "quotation": "quote",
    "storms": "storm", "cyclone": "storm", "cyclones": "storm", "typhoon": "storm", "hurricane": "storm",
    "risky": "risk", "risks": "risk", "dangerous": "risk", "riskiest": "risk",
    "top": "best", "ranked": "best", "ranking": "best", "rank": "best", "rankings": "best",
    "maximum": "highest", "max": "highest", "most": "highest", "biggest": "highest", "greatest": "highest",
    "moving": "transit", "enroute": "transit", "sailing": "transit",
    "margins": "margin", "profit": "margin", "profits": "margin",
    "many": "count", "number": "count", "total": "count", "count": "count",
    "quickest": "fastest", "shortest": "fastest", "faster": "fastest",
    "avg": "average", "mean": "average",
    "under": "below", "less": "below", "lower": "below", "low": "below",
    "heavy": "high", "severe": "high",
    "recent": "latest", "newest": "latest", "last": "latest",
    "mine": "my", "own": "my",
    "weather": "weather", "climate": "weather",
}

# Each intent: id, cap (RBAC), patterns (canonical wording), sql, optional entity + sql_entity, say template, example
INTENTS = [
    dict(id="cheapest_carrier_port", cap="carriers.view", entity="port",
         patterns=["cheapest carrier", "carrier cheapest cost", "cheapest carrier port"],
         sql="SELECT carrier, port, rate_usd FROM carrier_rates ORDER BY rate_usd ASC LIMIT 3",
         sql_entity="SELECT carrier, port, rate_usd FROM carrier_rates WHERE port = ? ORDER BY rate_usd ASC LIMIT 3",
         say="{carrier} offers the cheapest rate{to_port}: ${rate_usd:,.0f} per TEU.",
         example="Which carrier is cheapest to Rotterdam?"),
    dict(id="customs_hold_count", cap="customs.view",
         patterns=["count shipment customs hold", "count customs hold"],
         sql="SELECT COUNT(*) AS shipments_on_customs_hold FROM shipments WHERE customs_status = 'On Hold'",
         say="{shipments_on_customs_hold} shipment(s) are currently on customs hold.",
         example="How many shipments are on customs hold?"),
    dict(id="low_margin_quotes", cap="margin.view",
         patterns=["quote below margin", "below margin quote", "quote margin below"],
         sql="SELECT id, customer_id, origin, destination, carrier, margin_pct FROM quotes "
             "WHERE margin_pct < 10 {CUST} ORDER BY margin_pct ASC",
         say="{n} quote(s) have a margin below 10%.",
         example="Show quotes below 10% margin."),
    dict(id="congested_ports", cap="route.view",
         patterns=["highest congestion port", "port highest congestion", "congestion port"],
         sql="SELECT name AS port, country, congestion_index, congestion_level FROM ports "
             "ORDER BY congestion_index DESC LIMIT 5",
         say="{port} is the most congested port (congestion index {congestion_index}).",
         example="Which ports are most congested?"),
    dict(id="storms_region", cap="weather.view", entity="region",
         patterns=["storm", "storm weather", "weather storm"],
         sql="SELECT name, region, severity, wind_kts FROM storms ORDER BY wind_kts DESC",
         sql_entity="SELECT name, region, severity, wind_kts FROM storms WHERE region LIKE ? ORDER BY wind_kts DESC",
         say="{n} storm(s) found{in_region}. Strongest: {name} ({severity}, {wind_kts} kts).",
         example="Show storms near the Red Sea."),
    dict(id="highest_risk_carrier", cap="carriers.view",
         patterns=["highest risk carrier", "carrier highest risk", "risk carrier"],
         sql="SELECT name AS carrier, risk_score, on_time_pct FROM carriers ORDER BY risk_score DESC LIMIT 3",
         say="{carrier} has the highest risk score ({risk_score}/10).",
         example="Which carrier has the highest risk?"),
    dict(id="list_routes", cap="route.view",
         patterns=["route", "route list"],
         sql="SELECT origin, destination, via, cost_usd, transit_days FROM routes ORDER BY origin",
         say="{n} routes are available.",
         example="Show available shipping routes."),
    dict(id="lowest_cost_route", cap="route.view",
         patterns=["cheapest route", "cheapest route cost", "route cheapest cost"],
         sql="SELECT origin, destination, via, cost_usd, transit_days FROM routes ORDER BY cost_usd ASC LIMIT 3",
         say="The lowest-cost route is {origin} to {destination} via {via} at ${cost_usd:,.0f}.",
         example="Which route has the lowest cost?"),
    dict(id="quotes_count", cap="quotes.view",
         patterns=["count quote", "quote count"],
         sql="SELECT COUNT(*) AS total_quotes FROM quotes WHERE 1=1 {CUST}",
         say="{total_quotes} quote(s) have been generated.",
         example="How many quotes are generated?"),
    dict(id="weather_risk_shipments", cap="weather.view",
         patterns=["shipment weather risk", "weather risk shipment", "weather shipment"],
         sql="SELECT id, origin, destination, carrier, status, weather_risk FROM shipments "
             "WHERE weather_risk <> 'None' ORDER BY weather_risk ASC, id",
         say="{n} shipment(s) are exposed to weather risk.",
         example="Show shipments with weather risks."),
    dict(id="best_ranked_carrier", cap="carriers.view",
         patterns=["best carrier", "carrier best", "best carrier rank"],
         sql="SELECT name AS carrier, rank, risk_score, on_time_pct FROM carriers ORDER BY rank ASC LIMIT 3",
         say="{carrier} has the best ranking (#{rank}).",
         example="Which carrier has the best ranking?"),
    dict(id="delayed_shipments", cap="route.view",
         patterns=["delayed shipment", "shipment delayed"],
         sql="SELECT id, origin, destination, carrier, eta FROM shipments WHERE status = 'Delayed' ORDER BY eta",
         say="{n} shipment(s) are delayed.",
         example="Show the delayed shipments."),
    dict(id="in_transit_count", cap="route.view",
         patterns=["count shipment transit", "shipment transit count", "count transit"],
         sql="SELECT COUNT(*) AS shipments_in_transit FROM shipments WHERE status = 'In Transit'",
         say="{shipments_in_transit} shipment(s) are in transit.",
         example="How many shipments are in transit?"),
    dict(id="high_congestion_ports", cap="route.view",
         patterns=["high congestion port", "port high congestion"],
         sql="SELECT name AS port, country, congestion_index FROM ports WHERE congestion_level = 'High' "
             "ORDER BY congestion_index DESC",
         say="{n} port(s) have high congestion.",
         example="Which ports have high congestion?"),
    dict(id="customs_shipments", cap="customs.view",
         patterns=["customs shipment", "shipment customs hold", "customs"],
         sql="SELECT id, origin, destination, carrier, status, customs_status FROM shipments "
             "WHERE customs_status IN ('On Hold','Pending') ORDER BY customs_status, id",
         say="{n} shipment(s) have customs holds or pending clearance.",
         example="Show customs-related shipment information."),
    dict(id="shipments_by_status", cap="route.view",
         patterns=["shipment status", "count shipment status", "shipment summary"],
         sql="SELECT status, COUNT(*) AS shipments FROM shipments GROUP BY status ORDER BY shipments DESC",
         say="Shipments are grouped into {n} status categories.",
         example="Show shipment status summary."),
    dict(id="avg_margin", cap="margin.view",
         patterns=["average margin", "margin average", "average quote margin"],
         sql="SELECT ROUND(AVG(margin_pct), 1) AS avg_margin_pct FROM quotes WHERE 1=1 {CUST}",
         say="The average quote margin is {avg_margin_pct}%.",
         example="What is the average margin?"),
    dict(id="fastest_route", cap="route.view",
         patterns=["fastest route", "route fastest"],
         sql="SELECT origin, destination, via, transit_days, cost_usd FROM routes ORDER BY transit_days ASC LIMIT 3",
         say="The fastest route is {origin} to {destination} in {transit_days} days.",
         example="Which route is the fastest?"),
    dict(id="carrier_list", cap="carriers.view",
         patterns=["carrier", "carrier list"],
         sql="SELECT name AS carrier, cost_per_teu, risk_score, rank, on_time_pct FROM carriers ORDER BY rank",
         say="{n} carriers are on file.",
         example="Show all carriers."),
    dict(id="latest_quotes", cap="quotes.view",
         patterns=["latest quote", "my quote", "quote", "quote list"],
         sql="SELECT id, origin, destination, carrier, price, created_at FROM quotes WHERE 1=1 {CUST} "
             "ORDER BY created_at DESC LIMIT 10",
         say="Here are the latest {n} quote(s).",
         example="Show my latest quotes."),
]
INTENTS_BY_ID = {i["id"]: i for i in INTENTS}

Overwriting analyst.py


In [33]:
%%writefile intent_router.py
"""intent_router.py - spelling correction, synonym handling, entity detection and intent scoring."""
import re, difflib
import db
from analyst import INTENTS, STOP, SYNONYMS

GREET_WORDS = set("hi hello hey hola namaste greetings thanks thank thankyou good morning evening afternoon yo "
                  "bye there bot copilot howdy sup how are you u doing".split())
THRESHOLD = 0.6


def words(text):
    return re.sub(r"[^a-z0-9\s]", " ", (text or "").lower()).split()


def load_entities():
    ports = {r["name"].lower(): r["name"] for r in db.query("SELECT name FROM ports")}
    regions = {r["region"].lower(): r["region"] for r in db.query("SELECT DISTINCT region FROM storms")}
    return ports, regions


def build_vocab(ports, regions):
    vocab = set(STOP) | set(SYNONYMS) | set(SYNONYMS.values()) | GREET_WORDS
    for it in INTENTS:
        for p in it["patterns"]:
            vocab |= set(words(p))
    for name in list(ports) + list(regions):
        vocab |= set(words(name))
    return vocab


def correct(tokens, vocab):
    out, vl = [], sorted(vocab)
    for t in tokens:
        if t in vocab or len(t) <= 3 or t.isdigit():
            out.append(t)
        else:
            m = difflib.get_close_matches(t, vl, n=1, cutoff=0.75)
            out.append(m[0] if m else t)
    return out


def canon(tokens):
    res = []
    for t in tokens:
        if t in STOP or t[0].isdigit():
            continue
        t = SYNONYMS.get(t, t)
        if t not in res:
            res.append(t)
    return res


def pattern_tokens(p):
    return set(canon(words(p)))


def f_score(q, p):
    inter = q & p
    if not inter:
        return 0.0
    r, pr = len(inter) / len(p), len(inter) / len(q)
    return 2 * r * pr / (r + pr)


def find_entity(text, table):
    for key in sorted(table, key=len, reverse=True):
        if re.search(r"\b" + re.escape(key) + r"\b", text):
            return table[key], key
    return None, None


def route(question):
    ports, regions = load_entities()
    vocab = build_vocab(ports, regions)
    raw = words(question)
    fixed = correct(raw, vocab)
    text = " ".join(fixed)
    port, pkey = find_entity(text, ports)
    region, rkey = find_entity(text, regions)
    rest = text
    for k in (pkey, rkey):
        if k:
            rest = re.sub(r"\b" + re.escape(k) + r"\b", " ", rest)
    q = set(canon(rest.split()))
    best, best_score = None, 0.0
    for it in INTENTS:
        s = max(f_score(q, pattern_tokens(p)) for p in it["patterns"])
        if s > best_score:
            best, best_score = it, s
    interpreted = text if fixed != raw else None
    if best is not None and best_score >= THRESHOLD:
        ent = {"port": port, "region": region}.get(best.get("entity"))
        return dict(kind="intent", intent=best, entity=ent, score=round(best_score, 2), interpreted=interpreted)
    if raw and all(t in GREET_WORDS or t in STOP for t in fixed):
        return dict(kind="greeting")
    return dict(kind="unknown", interpreted=interpreted)

Overwriting intent_router.py


In [34]:
%%writefile qa.py
"""qa.py - runs whitelisted SQL for the routed intent, respecting RBAC and customer scoping."""
import random
import db, rbac
from analyst import INTENTS, INTENTS_BY_ID
from intent_router import route

MAX_ROWS = 50


def allowed_intents(user):
    return [i for i in INTENTS if rbac.has_cap(user, i["cap"])]


def suggest(user, n=3):
    pool = allowed_intents(user)
    return [i["example"] for i in random.sample(pool, min(n, len(pool)))]


def show_sql(sql, params):
    out = sql
    for p in params:
        out = out.replace("?", repr(p) if isinstance(p, str) else str(p), 1)
    return " ".join(out.split())


def reply(text, **kw):
    base = dict(answer=text, sql=None, row_count=0, columns=[], rows=[], interpreted=None, suggestions=[])
    base.update(kw)
    return base


def ask(user, question):
    question = (question or "").strip()
    if not question:
        return reply("Please type a question.", suggestions=suggest(user))
    r = route(question)
    if r["kind"] == "greeting":
        return reply(f"Hello {user['username']}! I'm your FreightQuote AI Copilot. "
                     "I answer from live database queries. Try one of these:", suggestions=suggest(user))
    if r["kind"] == "unknown":
        return reply("Sorry, I can't answer that from the data I have access to. "
                     "Here are three questions I can answer:", suggestions=suggest(user), interpreted=r["interpreted"])
    it = r["intent"]
    if not rbac.has_cap(user, it["cap"]):
        return reply(f"Your role ({user['role']}) isn't allowed to see that data. "
                     "Here are questions I can answer for you:", suggestions=suggest(user), denied=True)
    ent = r["entity"]
    sql, params = (it["sql_entity"], [f"%{ent}%" if it["entity"] == "region" else ent]) if ent and it.get("sql_entity") \
        else (it["sql"], [])
    if "{CUST}" in sql:
        if user["role"] == "Customer":
            sql, params = sql.replace("{CUST}", "AND customer_id = ?"), params + [rbac.customer_scope(user) or ""]
        else:
            sql = sql.replace("{CUST}", "")
    rows = db.query(sql, params)[:MAX_ROWS]
    cols = list(rows[0].keys()) if rows else []
    if rows:
        ctx = dict(rows[0], n=len(rows), to_port=f" to {ent}" if it.get("entity") == "port" and ent else "",
                   in_region=f" near {ent}" if it.get("entity") == "region" and ent else "")
        try:
            text = it["say"].format(**ctx)
        except Exception:
            text = f"Found {len(rows)} row(s)."
    else:
        text = "No matching records were found for that question."
    return reply(text, sql=show_sql(sql, params), row_count=len(rows), columns=cols, rows=rows,
                 interpreted=r["interpreted"], intent=it["id"])

Overwriting qa.py


In [35]:
%%writefile main.py
"""main.py - FreightQuote AI (combined): auth (M1) + RBAC (M2) + Admin dashboard (M3) + Copilot & UI (M4)."""
import os
import sqlite3

from fastapi import Depends, FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

import auth
import db
import qa
import rbac
import security as sec
import agent_engine
import agent_worker

db.init_db()
# Safe additive migration for existing databases (SCHEMA creates these if absent).
os.makedirs("static", exist_ok=True)
app = FastAPI(title="FreightQuote AI")
app.include_router(auth.router)
app.mount("/static", StaticFiles(directory="static"), name="static")


def log(actor, action, detail=""):
    db.execute("INSERT INTO activity_log(actor,action,detail,ts) VALUES(?,?,?,?)", (actor["email"], action, detail, db.now()))


@app.get("/")
def index():
    return FileResponse("static/index.html")


@app.get("/milestone3")
def milestone3_dashboard():
    return FileResponse("static/milestone3.html")


@app.get("/api/dashboard/summary")
def dashboard_summary(user: dict = Depends(auth.user_of)):
    # Dashboard data is scoped to the authenticated session.
    shipments = db.query("SELECT COUNT(*) AS total, SUM(CASE WHEN LOWER(weather_risk)='high' THEN 1 ELSE 0 END) AS high_risk FROM shipments", one=True) or {}
    quotes = db.query("SELECT COUNT(*) AS total, COALESCE(SUM(price-cost),0) AS profit FROM quotes", one=True) or {}
    agents = db.query("SELECT agent,status,summary,updated_at FROM agent_results ORDER BY agent")
    runs = db.query("SELECT started_at,finished_at,status,summary FROM agent_runs ORDER BY id DESC LIMIT 5")
    activity = db.query("SELECT actor,action,detail,ts FROM activity_log ORDER BY id DESC LIMIT 6")
    routes = db.query("SELECT COUNT(*) AS total FROM route_decisions", one=True) or {}
    storms = db.query("SELECT name,region,severity,wind_kts FROM storms ORDER BY id DESC LIMIT 1", one=True)
    return {"shipments": shipments, "quotes": quotes, "agents": agents, "runs": runs,
            "activity": activity, "routes": routes, "latest_storm": storms,
            "worker_interval_seconds": int(os.environ.get("AGENT_INTERVAL_SECONDS", "300"))}


@app.get("/api/health")
def health():
    return {"status": "ok"}

@app.on_event("startup")
def start_background_agents():
    agent_worker.start_worker()


@app.get("/api/ping")
def ping(user: dict = Depends(auth.user_of)):
    return {"message": f"Authenticated as {user['username']}", "role": user["role"]}


@app.get("/api/menu")
def menu(user: dict = Depends(auth.user_of)):
    return {"role": user["role"], "pages": rbac.menu_for(user), "caps": rbac.caps_for(user)}


# ---------------- RBAC-protected data pages ----------------
PAGES = {
    "route": ("route.view", "SELECT s.id AS shipment_id, s.origin, s.destination, s.weather_risk, rd.via, rd.cost_usd, rd.transit_days, rd.reason FROM shipments s LEFT JOIN route_decisions rd ON rd.shipment_id=s.id ORDER BY s.id"),
    "quotes": ("quotes.view", "SELECT id, customer_id, origin, destination, container, carrier, price, created_at FROM quotes WHERE 1=1 {CUST} ORDER BY id DESC"),
    "carriers": ("carriers.view", "SELECT name AS carrier, cost_per_teu, risk_score, rank, on_time_pct FROM carriers ORDER BY rank"),
    "weather": ("weather.view", "SELECT name, region, severity, wind_kts FROM storms ORDER BY wind_kts DESC"),
    "margin": ("margin.view", "SELECT id, carrier, cost, price, margin_pct FROM quotes ORDER BY margin_pct ASC"),
    "customs": ("customs.view", "SELECT id, origin, destination, carrier, status, customs_status FROM shipments ORDER BY customs_status, id"),
}


@app.get("/api/page/{name}")
def page(name: str, user: dict = Depends(auth.user_of)):
    if name not in PAGES:
        raise HTTPException(404, "Unknown page.")
    cap, sql = PAGES[name]
    rbac.need_page(user, name)
    rbac.need(user, cap)
    params = []
    if "{CUST}" in sql:
        scope = rbac.customer_scope(user)
        if scope:
            sql, params = sql.replace("{CUST}", "AND customer_id = ?"), [scope]
        else:
            sql = sql.replace("{CUST}", "")
    rows = db.query(sql, params)
    return {"columns": list(rows[0].keys()) if rows else [], "rows": rows}


# ---------------- role-limited actions (Member 2) ----------------
@app.post("/api/route/reroute")
def reroute(user: dict = Depends(rbac.require("route.reroute"))):
    result = agent_engine.reroute_shipments()
    log(user, "reroute", f"recalculated {result['shipments_checked']} shipments; weather avoidance {result['weather_avoidance_reroutes']}")
    return {"message": "Route recalculation completed.", "by": user["username"], **result}


@app.post("/api/weather/storm")
def inject_storm(user: dict = Depends(rbac.require("weather.inject_storm"))):
    n = db.query("SELECT COUNT(*) AS n FROM storms", one=True)["n"] + 1
    name = f"Storm Injected-{n}"
    db.execute("INSERT INTO storms(name,region,severity,wind_kts) VALUES(?,?,?,?)", (name, "Red Sea", "High", 80))
    weather_result = agent_engine.refresh_weather()
    reroute_result = agent_engine.reroute_shipments()
    log(user, "inject_storm", f"{name}; weather refreshed; {reroute_result['weather_avoidance_reroutes']} weather reroutes")
    return {"message": f"{name} injected; weather risk refreshed and routes recalculated.",
            "by": user["username"], "weather_agent": weather_result,
            "reroute": reroute_result}


@app.post("/api/customs/edit")
def edit_customs(user: dict = Depends(rbac.require("customs.edit"))):
    row = db.query("SELECT id, customs_status FROM shipments ORDER BY id LIMIT 1", one=True)
    new = "Cleared" if row["customs_status"] == "On Hold" else "On Hold"
    db.execute("UPDATE shipments SET customs_status=? WHERE id=?", (new, row["id"]))
    log(user, "edit_customs", f"{row['id']} -> {new}")
    return {"message": f"{row['id']} is now '{new}'.", "by": user["username"]}


@app.post("/api/agents/run")
def run_agents(user: dict = Depends(rbac.require("agents.run"))):
    results = agent_engine.run_all_agents()
    log(user, "run_agents", "six agents executed")
    return {"message": "Six agents executed.", "by": user["username"], "results": results}

@app.get("/api/agents/results")
def agent_results(user: dict = Depends(auth.user_of)):
    rbac.need(user, "route.view")
    rows = db.query("SELECT agent,status,summary,payload,updated_at FROM agent_results ORDER BY agent")
    for row in rows:
        try: row["payload"] = __import__("json").loads(row["payload"] or "{}")
        except Exception: row["payload"] = {}
    return {"results": rows}

@app.post("/api/agents/weather/refresh")
def weather_refresh(user: dict = Depends(rbac.require("agents.run"))):
    result = agent_engine.refresh_weather()
    log(user, "weather_refresh", result["summary"])
    return result

@app.get("/api/routes/decisions")
def route_decisions(user: dict = Depends(rbac.require("route.view"))):
    return {"routes": db.query("SELECT s.id AS shipment_id,s.origin,s.destination,s.weather_risk,rd.via,rd.cost_usd,rd.transit_days,rd.reason,rd.updated_at FROM shipments s LEFT JOIN route_decisions rd ON rd.shipment_id=s.id ORDER BY s.id")}

@app.get("/api/agents/runs")
def agent_runs(user: dict = Depends(rbac.require("route.view"))):
    return {"runs": db.query("SELECT * FROM agent_runs ORDER BY id DESC LIMIT 20")}


# ---------------- quotes (customers see only their own) ----------------
CONTAINERS = {"20ft": 1.0, "40ft": 1.8, "40ft HC": 1.9}


class QuoteIn(BaseModel):
    origin: str = ""
    destination: str = ""
    container: str = "40ft"
    customer_id: str = ""


@app.get("/api/quotes")
def list_quotes(user: dict = Depends(rbac.require("quotes.view"))):
    scope = rbac.customer_scope(user)
    if scope:
        return {"quotes": db.query("SELECT * FROM quotes WHERE customer_id=? ORDER BY id DESC", (scope,))}
    return {"quotes": db.query("SELECT * FROM quotes ORDER BY id DESC")}


@app.get("/api/quotes/{qid}")
def get_quote(qid: int, user: dict = Depends(rbac.require("quotes.view"))):
    row = db.query("SELECT * FROM quotes WHERE id=?", (qid,), one=True)
    if not row:
        raise HTTPException(404, "Quote not found.")
    scope = rbac.customer_scope(user)
    if scope and row["customer_id"] != scope:
        raise HTTPException(403, "Access denied (403): this quote belongs to another customer.")
    return row


@app.post("/api/quotes", status_code=201)
def create_quote(body: QuoteIn, user: dict = Depends(rbac.require("quotes.create"))):
    origin, dest = body.origin.strip()[:60], body.destination.strip()[:60]
    if not origin or not dest:
        raise HTTPException(400, "Origin and destination are required.")
    if body.container not in CONTAINERS:
        raise HTTPException(400, "Container must be one of: " + ", ".join(CONTAINERS) + ".")
    scope = rbac.customer_scope(user)
    cid = scope or body.customer_id.strip()[:20]  # customers can only create for themselves
    if not cid:
        raise HTTPException(400, "customer_id is required.")
    carrier = db.query("SELECT name, cost_per_teu FROM carriers ORDER BY cost_per_teu ASC LIMIT 1", one=True)
    route = db.query("SELECT cost_usd FROM routes WHERE LOWER(origin)=LOWER(?) AND LOWER(destination)=LOWER(?)", (origin, dest), one=True)
    base = float(route["cost_usd"]) if route else float(carrier["cost_per_teu"])
    cost = round(base * CONTAINERS[body.container])
    margin = 12.5
    price = round(cost / (1 - margin / 100), 2)
    qid = db.execute(
        "INSERT INTO quotes(customer_id,origin,destination,container,carrier,cost,price,margin_pct,created_by,created_at) "
        "VALUES(?,?,?,?,?,?,?,?,?,?)",
        (cid, origin, dest, body.container, carrier["name"], cost, price, margin, user["username"], db.now()))
    log(user, "create_quote", f"#{qid} for {cid}")
    return {"message": "Quote created.", "quote": db.query("SELECT * FROM quotes WHERE id=?", (qid,), one=True)}


# ---------------- Copilot (Member 4) ----------------
class Ask(BaseModel):
    question: str = ""


@app.post("/api/copilot")
def copilot(b: Ask, user: dict = Depends(rbac.require("copilot.use"))):
    return qa.ask(user, b.question)


@app.get("/api/copilot/suggestions")
def suggestions(user: dict = Depends(rbac.require("copilot.use"))):
    return {"suggestions": qa.suggest(user, 6)}


# ---------------- Admin dashboard (Member 3) ----------------
admin_only = rbac.require("users.manage")


def admin_count():
    return db.query("SELECT COUNT(*) AS n FROM users WHERE role='Admin'", one=True)["n"]


def target_of(uid):
    t = db.query("SELECT * FROM users WHERE id = ?", (uid,), one=True)
    if not t:
        raise HTTPException(404, "User not found.")
    return t


@app.get("/api/admin/users")
def admin_users(admin: dict = Depends(admin_only)):
    rows = db.query("SELECT id,email,username,role,locked,customer_id,created_at,last_login FROM users ORDER BY id")
    for r in rows:
        r["status"] = "Locked" if r["locked"] else "Active"
    return {"users": rows, "roles": rbac.ROLES}


@app.get("/api/admin/kpis")
def admin_kpis(admin: dict = Depends(admin_only)):
    by_role = {r: 0 for r in rbac.ROLES}
    for r in db.query("SELECT role, COUNT(*) AS n FROM users GROUP BY role"):
        by_role[r["role"]] = r["n"]
    return {"total": sum(by_role.values()), "by_role": by_role,
            "locked": db.query("SELECT COUNT(*) AS n FROM users WHERE locked=1", one=True)["n"]}


@app.get("/api/admin/activity")
def admin_activity(admin: dict = Depends(admin_only)):
    return {"activity": db.query("SELECT actor, action, detail, ts FROM activity_log ORDER BY id DESC LIMIT 50")}


class NewUser(BaseModel):
    email: str = ""
    username: str = ""
    password: str = ""
    role: str = "Customer"


@app.post("/api/admin/users")
def add_user(b: NewUser, admin: dict = Depends(admin_only)):
    email, uname = b.email.strip().lower(), b.username.strip()
    if not sec.valid_email(email):
        raise HTTPException(400, "Enter a valid email address.")
    if not sec.valid_username(uname):
        raise HTTPException(400, "Username must be 3-30 characters: letters, numbers, dot, dash or underscore.")
    if b.role not in rbac.ROLES:
        raise HTTPException(400, "Choose a valid role.")
    problems = sec.password_problems(b.password)
    if problems:
        raise HTTPException(400, "Password must contain " + ", ".join(problems) + ".")
    cust = db.next_customer_id() if b.role == "Customer" else None
    try:
        db.execute("INSERT INTO users(email,username,password_hash,role,customer_id,created_at) VALUES(?,?,?,?,?,?)",
                   (email, uname, sec.hash_password(b.password), b.role, cust, db.now()))
    except sqlite3.IntegrityError:
        raise HTTPException(400, "That email or username is already registered.")
    log(admin, "add_user", f"{email} as {b.role}")
    return {"ok": True}


@app.delete("/api/admin/users/{uid}")
def delete_user(uid: int, admin: dict = Depends(admin_only)):
    t = target_of(uid)
    if t["id"] == admin["id"]:
        raise HTTPException(400, "You cannot delete your own account.")
    if t["role"] == "Admin" and admin_count() <= 1:
        raise HTTPException(400, "The last remaining Admin cannot be deleted.")
    db.execute("DELETE FROM users WHERE id=?", (uid,))
    log(admin, "delete_user", t["email"])
    return {"ok": True}


class RoleChange(BaseModel):
    role: str = ""


@app.post("/api/admin/users/{uid}/role")
def change_role(uid: int, b: RoleChange, admin: dict = Depends(admin_only)):
    t = target_of(uid)
    if b.role not in rbac.ROLES:
        raise HTTPException(400, "Choose a valid role.")
    if t["id"] == admin["id"]:
        raise HTTPException(400, "You cannot change your own role.")
    if t["role"] == "Admin" and b.role != "Admin" and admin_count() <= 1:
        raise HTTPException(400, "The last remaining Admin cannot be demoted.")
    cust = t["customer_id"] or (db.next_customer_id() if b.role == "Customer" else None)
    db.execute("UPDATE users SET role=?, customer_id=? WHERE id=?", (b.role, cust, uid))
    log(admin, "change_role", f"{t['email']}: {t['role']} -> {b.role}")
    return {"ok": True}


@app.post("/api/admin/users/{uid}/lock")
def lock_user(uid: int, admin: dict = Depends(admin_only)):
    t = target_of(uid)
    if t["id"] == admin["id"]:
        raise HTTPException(400, "You cannot lock your own account.")
    db.execute("UPDATE users SET locked=1 WHERE id=?", (uid,))
    log(admin, "lock_user", t["email"])
    return {"ok": True}


@app.post("/api/admin/users/{uid}/unlock")
def unlock_user(uid: int, admin: dict = Depends(admin_only)):
    t = target_of(uid)
    db.execute("UPDATE users SET locked=0, failed_attempts=0 WHERE id=?", (uid,))
    log(admin, "unlock_user", t["email"])
    return {"ok": True}

Overwriting main.py


In [36]:
%%writefile static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>FreightQuote AI</title>
<link rel="stylesheet" href="/static/style.css">
</head>
<body>
<div id="loader"><div class="spinner"></div></div>
<div id="toasts" aria-live="polite"></div>
<button id="themeBtn" class="icon-btn theme-fixed" title="Toggle dark mode" aria-label="Toggle dark mode">🌙</button>

<main id="auth" hidden>
  <div class="auth-card">
    <div class="brand">🚢 FreightQuote <span>AI</span></div>
    <div class="tabs"><button class="tab active" data-tab="login">Login</button><button class="tab" data-tab="signup">Sign up</button></div>
    <form id="loginForm" class="form">
      <label>Email<input name="email" type="email" autocomplete="username" required placeholder="you@company.com"></label>
      <label>Password<input name="password" type="password" autocomplete="current-password" required></label>
      <button class="btn primary" type="submit">Sign in</button>
      <p class="small"><a href="#" id="toForgot">Forgot password?</a></p>
      <p class="muted small">Demo: admin@freightquote.ai / Admin@123</p>
    </form>
    <form id="signupForm" class="form" hidden>
      <label>Email<input name="email" type="email" autocomplete="email" required></label>
      <label>Username<input name="username" autocomplete="username" required minlength="3" maxlength="30"></label>
      <label>Password<input name="password" type="password" autocomplete="new-password" required></label>
      <label>Confirm password<input name="confirm_password" type="password" autocomplete="new-password" required></label>
      <p class="muted small">8-64 characters with upper, lower, digit and special character.</p>
      <button class="btn primary" type="submit">Create account</button>
    </form>
    <form id="forgotForm" class="form" hidden>
      <p class="muted small">We will email you a 6-digit reset code (printed in the Colab output if email is not configured).</p>
      <label>Email<input name="email" type="email" required></label>
      <button class="btn primary" type="submit">Send reset code</button>
      <p class="small"><a href="#" class="toLogin">Back to login</a> · <a href="#" id="toReset">I already have a code</a></p>
    </form>
    <form id="resetForm" class="form" hidden>
      <label>Email<input name="email" type="email" required></label>
      <label>Reset code<input name="code" inputmode="numeric" maxlength="6" required></label>
      <label>New password<input name="new_password" type="password" autocomplete="new-password" required></label>
      <label>Confirm new password<input name="confirm_password" type="password" autocomplete="new-password" required></label>
      <button class="btn primary" type="submit">Reset password</button>
      <p class="small"><a href="#" class="toLogin">Back to login</a></p>
    </form>
  </div>
</main>

<div id="shell" hidden>
  <aside id="sidebar">
    <div class="brand">FreightQuote <span>AI</span></div>
    <nav id="nav"></nav>
  </aside>
  <div id="backdrop"></div>
  <section class="content">
    <header class="topbar">
      <button id="menuBtn" class="icon-btn" aria-label="Menu">☰</button>
      <h1 id="pageTitle"></h1>
      <div class="spacer"></div>
      <span id="userChip" class="chip"></span>
      <button id="signOut" class="btn small">Sign out</button>
    </header>
    <div id="view"></div>
  </section>
</div>
<script src="/static/app.js"></script>
</body>
</html>

Overwriting static/index.html


In [37]:
%%writefile static/style.css
:root{--bg:#f4f6fb;--card:#fff;--text:#1b2433;--muted:#6b7686;--border:#e1e6ef;--primary:#1f6feb;--primary-ink:#fff;
--accent:#0ea5a4;--danger:#d92d20;--ok:#12805c;--warn:#b54708;--shadow:0 2px 10px rgba(20,30,60,.07);--row:#f8faff}
[data-theme="dark"]{--bg:#0e1420;--card:#171f2e;--text:#e6ebf5;--muted:#93a0b5;--border:#27324a;--primary:#4c8dff;--primary-ink:#06101f;
--accent:#2dd4bf;--danger:#ff6b61;--ok:#4ade80;--warn:#fbbf24;--shadow:0 2px 12px rgba(0,0,0,.4);--row:#1b2536}
*{box-sizing:border-box}
html,body{margin:0;height:100%}
body{font-family:system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;background:var(--bg);color:var(--text);line-height:1.45}
[hidden]{display:none!important}
.muted{color:var(--muted)}.small{font-size:.85rem}
.brand{font-weight:800;font-size:1.25rem;letter-spacing:.2px}.brand span{color:var(--primary)}
.icon-btn{background:var(--card);border:1px solid var(--border);color:var(--text);border-radius:10px;width:40px;height:40px;font-size:1.1rem;cursor:pointer}
.theme-fixed{position:fixed;top:12px;right:12px;z-index:60}
.btn{border:1px solid var(--border);background:var(--card);color:var(--text);padding:.55rem .95rem;border-radius:10px;cursor:pointer;font-size:.95rem;font-weight:600}
.btn:hover{border-color:var(--primary)}.btn.small{padding:.35rem .7rem;font-size:.85rem}
.btn.primary{background:var(--primary);color:var(--primary-ink);border-color:var(--primary)}
.btn.danger{color:var(--danger);border-color:var(--danger)}
input,select,textarea{width:100%;padding:.6rem .75rem;border:1px solid var(--border);border-radius:10px;background:var(--bg);color:var(--text);font-size:1rem}
input:focus,select:focus{outline:2px solid var(--primary);outline-offset:0}
label{display:block;font-weight:600;font-size:.9rem}label input,label select{margin-top:.3rem;font-weight:400}
/* auth */
#auth{min-height:100%;display:grid;place-items:center;padding:1rem;background:radial-gradient(circle at 20% 10%,rgba(31,111,235,.18),transparent 45%),var(--bg)}
.auth-card{width:100%;max-width:400px;background:var(--card);border:1px solid var(--border);border-radius:18px;padding:1.6rem;box-shadow:var(--shadow)}
.auth-card .brand{text-align:center;margin-bottom:1rem}
.tabs{display:flex;gap:.4rem;background:var(--bg);padding:.3rem;border-radius:12px;margin-bottom:1rem}
.tab{flex:1;border:0;background:transparent;color:var(--muted);padding:.5rem;border-radius:9px;font-weight:700;cursor:pointer}
.tab.active{background:var(--card);color:var(--text);box-shadow:var(--shadow)}
.form{display:grid;gap:.85rem}
/* shell */
#shell{display:flex;min-height:100%}
#sidebar{width:240px;flex:none;background:var(--card);border-right:1px solid var(--border);padding:1.1rem .8rem;position:sticky;top:0;height:100vh;overflow:auto}
#sidebar .brand{padding:0 .5rem 1rem}
#nav{display:grid;gap:.25rem}
#nav button{display:flex;gap:.6rem;align-items:center;text-align:left;border:0;background:transparent;color:var(--text);padding:.65rem .75rem;border-radius:10px;font-size:.98rem;cursor:pointer}
#nav button:hover{background:var(--row)}
#nav button.active{background:var(--primary);color:var(--primary-ink);font-weight:700}
.content{flex:1;min-width:0;display:flex;flex-direction:column}
.topbar{display:flex;align-items:center;gap:.75rem;padding:.8rem 1.2rem;background:var(--card);border-bottom:1px solid var(--border);position:sticky;top:0;z-index:20;padding-right:4rem}
.topbar h1{font-size:1.15rem;margin:0}.spacer{flex:1}#menuBtn{display:none}
.chip{background:var(--row);border:1px solid var(--border);border-radius:99px;padding:.25rem .7rem;font-size:.82rem}
#view{padding:1.2rem;max-width:1200px;width:100%}
.card{background:var(--card);border:1px solid var(--border);border-radius:14px;padding:1rem 1.1rem;box-shadow:var(--shadow);margin-bottom:1rem}
.card h2{margin:.1rem 0 .8rem;font-size:1.05rem}
.grid{display:grid;gap:.9rem;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));margin-bottom:1rem}
.kpi{background:var(--card);border:1px solid var(--border);border-radius:14px;padding:.9rem 1rem;box-shadow:var(--shadow)}
.kpi .n{font-size:1.9rem;font-weight:800}.kpi .l{color:var(--muted);font-size:.82rem}
.kpi.warn .n{color:var(--danger)}
.table-wrap{overflow-x:auto;border:1px solid var(--border);border-radius:12px}
table{border-collapse:collapse;width:100%;font-size:.92rem}
th,td{padding:.6rem .8rem;text-align:left;border-bottom:1px solid var(--border);white-space:nowrap}
th{background:var(--row);color:var(--muted);font-size:.76rem;text-transform:uppercase;letter-spacing:.05em;position:sticky;top:0}
tbody tr:nth-child(even){background:var(--row)}tbody tr:hover{background:rgba(76,141,255,.12)}
tr:last-child td{border-bottom:0}
td.num,th.num{text-align:right}
.badge{display:inline-block;padding:.12rem .55rem;border-radius:99px;font-size:.78rem;font-weight:700}
.badge.ok{background:rgba(18,128,92,.15);color:var(--ok)}.badge.bad{background:rgba(217,45,32,.15);color:var(--danger)}
.row{display:flex;flex-wrap:wrap;gap:.6rem;align-items:flex-end}.row>*{flex:1 1 160px}.row>.btn{flex:0 0 auto}
.actions{display:flex;gap:.4rem}
/* copilot */
.chat{display:flex;flex-direction:column;gap:.8rem;min-height:300px;max-height:58vh;overflow:auto;padding:.2rem}
.msg{max-width:92%;padding:.7rem .9rem;border-radius:14px;border:1px solid var(--border);background:var(--card)}
.msg.user{align-self:flex-end;background:var(--primary);color:var(--primary-ink);border-color:var(--primary)}
.msg.bot{align-self:flex-start}
.msg details{margin-top:.5rem}.msg summary{cursor:pointer;color:var(--muted);font-size:.85rem}
pre.sql{background:var(--bg);border:1px solid var(--border);border-radius:10px;padding:.6rem;overflow:auto;font-size:.82rem;white-space:pre-wrap;word-break:break-word}
.meta{font-size:.8rem;color:var(--muted);margin:.4rem 0}
.chips{display:flex;flex-wrap:wrap;gap:.45rem;margin-top:.55rem}
.chips button{border:1px solid var(--border);background:var(--bg);color:var(--text);border-radius:99px;padding:.3rem .75rem;cursor:pointer;font-size:.85rem}
.chips button:hover{border-color:var(--primary);color:var(--primary)}
.chat-input{display:flex;gap:.5rem;margin-top:.8rem}.chat-input input{flex:1}
.dots span{display:inline-block;width:7px;height:7px;margin:0 2px;border-radius:50%;background:var(--muted);animation:b 1s infinite}
.dots span:nth-child(2){animation-delay:.15s}.dots span:nth-child(3){animation-delay:.3s}
@keyframes b{0%,80%,100%{opacity:.25}40%{opacity:1}}
/* feedback */
#loader{position:fixed;inset:0;display:none;place-items:center;background:rgba(10,15,25,.25);z-index:80}
#loader.show{display:grid}
.spinner{width:42px;height:42px;border-radius:50%;border:4px solid var(--border);border-top-color:var(--primary);animation:s .8s linear infinite}
@keyframes s{to{transform:rotate(360deg)}}
#toasts{position:fixed;bottom:16px;left:50%;transform:translateX(-50%);display:grid;gap:.5rem;z-index:90;width:min(92vw,420px)}
.toast{padding:.7rem .95rem;border-radius:12px;color:#fff;box-shadow:var(--shadow);font-weight:600}
.toast.error{background:#c0392b}.toast.ok{background:#12805c}
.skeleton{height:90px;border-radius:12px;background:linear-gradient(90deg,var(--row),var(--border),var(--row));background-size:200% 100%;animation:sk 1.2s infinite}
@keyframes sk{to{background-position:-200% 0}}
/* mobile */
#backdrop{display:none}
@media(max-width:800px){
 #sidebar{position:fixed;left:0;top:0;z-index:70;transform:translateX(-100%);transition:transform .2s;box-shadow:var(--shadow)}
 body.menu-open #sidebar{transform:none}
 body.menu-open #backdrop{display:block;position:fixed;inset:0;background:rgba(0,0,0,.4);z-index:65}
 #menuBtn{display:inline-block}#userChip{display:none}
 #view{padding:.8rem}.msg{max-width:100%}
}
/* combined-app extras */
.auth-card a,.form a{color:var(--primary)}
.btn:disabled{opacity:.6;cursor:wait}
.card .actions{flex-wrap:wrap}

Overwriting static/style.css


In [38]:
%%writefile static/app.js
"use strict";
const $ = (s, r = document) => r.querySelector(s);
const esc = v => String(v ?? "").replace(/[&<>"']/g, c => ({"&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"}[c]));
const store = {get: k => { try { return localStorage.getItem(k); } catch (e) { return null; } },
               set: (k, v) => { try { localStorage.setItem(k, v); } catch (e) {} }};
let me = null, menu = [], caps = [], current = "copilot", pending = 0;
const ICON = {copilot: "🤖", route: "🧭", quotes: "💲", carriers: "🚢", weather: "🌦️", margin: "📈", customs: "🛃", admin: "🛡️", profile: "👤"};
const ACTIONS = {
  route: [{cap: "route.reroute", label: "Re-route", path: "/api/route/reroute"}],
  weather: [{cap: "weather.inject_storm", label: "Inject storm", path: "/api/weather/storm"}],
  customs: [{cap: "customs.edit", label: "Toggle customs hold", path: "/api/customs/edit"}],
};
const labelOf = p => p === "profile" ? "Profile" : ((menu.find(m => m.id === p) || {}).label || p);

/* ---------- theme ---------- */
function applyTheme(t) {
  document.documentElement.setAttribute("data-theme", t);
  $("#themeBtn").textContent = t === "dark" ? "☀️" : "🌙";
  store.set("fq-theme", t);
}
applyTheme(store.get("fq-theme") || (matchMedia("(prefers-color-scheme: dark)").matches ? "dark" : "light"));
$("#themeBtn").onclick = () => applyTheme(document.documentElement.getAttribute("data-theme") === "dark" ? "light" : "dark");

/* ---------- helpers ---------- */
function toast(msg, type = "ok") {
  const d = document.createElement("div");
  d.className = "toast " + type; d.textContent = msg;
  $("#toasts").appendChild(d);
  setTimeout(() => d.remove(), 4200);
}
function loading(on) { pending += on ? 1 : -1; $("#loader").classList.toggle("show", pending > 0); }

async function api(path, method = "GET", body = null, opts = {}) {
  if (!opts.silent) loading(true);
  try {
    let res;
    try {
      res = await fetch(path, {method, credentials: "same-origin", headers: body ? {"Content-Type": "application/json"} : {},
                               body: body ? JSON.stringify(body) : undefined});
    } catch (e) { throw new Error("Cannot reach the server. Check your connection and try again."); }
    let data = null; try { data = await res.json(); } catch (e) {}
    if (!res.ok) {
      let msg = "Request failed (" + res.status + ")";
      if (data && data.detail) msg = typeof data.detail === "string" ? data.detail : data.detail.map(d => d.msg).join("; ");
      const err = new Error(msg); err.status = res.status; throw err;
    }
    return data;
  } catch (e) {
    if ((e.status === 401 || e.status === 423) && me) signOut();
    if (!opts.quiet) toast(e.message, "error");
    throw e;
  } finally { if (!opts.silent) loading(false); }
}

function table(cols, rows, fmt = {}) {
  if (!rows.length) return '<p class="muted">No rows to show.</p>';
  const isNum = c => typeof rows[0][c] === "number";
  return `<div class="table-wrap"><table><thead><tr>${cols.map(c => `<th class="${isNum(c) ? "num" : ""}">${esc(c.replace(/_/g, " "))}</th>`).join("")}</tr></thead><tbody>` +
    rows.map(r => `<tr>${cols.map(c => `<td class="${isNum(c) ? "num" : ""}">${fmt[c] ? fmt[c](r[c], r) : esc(r[c])}</td>`).join("")}</tr>`).join("") +
    "</tbody></table></div>";
}
const fmtDate = iso => { if (!iso) return "—"; const d = new Date(iso); return isNaN(d) ? iso : d.toLocaleString(); };

/* ---------- auth screens ---------- */
const FORMS = ["login", "signup", "forgot", "reset"];
function showAuth(name) {
  FORMS.forEach(f => { $("#" + f + "Form").hidden = f !== name; });
  document.querySelectorAll(".tab").forEach(x => x.classList.toggle("active", x.dataset.tab === name));
}
function showAuthScreen() { $("#auth").hidden = false; $("#shell").hidden = true; showAuth("login"); }
document.querySelectorAll(".tab").forEach(b => b.onclick = () => showAuth(b.dataset.tab));
$("#toForgot").onclick = e => { e.preventDefault(); showAuth("forgot"); };
$("#toReset").onclick = e => { e.preventDefault(); showAuth("reset"); };
document.querySelectorAll(".toLogin").forEach(a => a.onclick = e => { e.preventDefault(); showAuth("login"); });

function wire(formId, handler) {
  const form = $("#" + formId);
  form.onsubmit = async ev => {
    ev.preventDefault();
    const btn = form.querySelector("button[type=submit]"), label = btn.textContent;
    btn.disabled = true; btn.textContent = "Please wait...";
    try { await handler(Object.fromEntries(new FormData(form)), form); } catch (e) { /* toast already shown */ }
    finally { btn.disabled = false; btn.textContent = label; }
  };
}
wire("loginForm", async (d, f) => { await api("/api/auth/login", "POST", d); f.reset(); await start(); });
wire("signupForm", async (d, f) => {
  if (d.password !== d.confirm_password) { toast("Passwords do not match.", "error"); return; }
  const r = await api("/api/auth/signup", "POST", d); toast(r.message); f.reset(); showAuth("login");
});
wire("forgotForm", async (d, f) => {
  const r = await api("/api/auth/forgot", "POST", d); toast(r.message);
  $("#resetForm").elements.email.value = d.email; showAuth("reset");
});
wire("resetForm", async (d, f) => {
  if (d.new_password !== d.confirm_password) { toast("Passwords do not match.", "error"); return; }
  const r = await api("/api/auth/reset", "POST", d); toast(r.message); f.reset(); showAuth("login");
});

function signOut() {
  const wasIn = !!me;
  me = null; menu = []; caps = []; current = "copilot";
  document.body.classList.remove("menu-open");
  if (wasIn) fetch("/api/auth/logout", {method: "POST", credentials: "same-origin"}).catch(() => {});
  showAuthScreen();
}
$("#signOut").onclick = signOut;
$("#menuBtn").onclick = () => document.body.classList.toggle("menu-open");
$("#backdrop").onclick = () => document.body.classList.remove("menu-open");

/* ---------- shell ---------- */
async function start() {
  try {
    me = (await api("/api/auth/me", "GET", null, {silent: true, quiet: true})).user;
    const m = await api("/api/menu", "GET", null, {silent: true, quiet: true});
    menu = m.pages; caps = m.caps;
  } catch (e) { me = null; showAuthScreen(); return; }
  $("#auth").hidden = true; $("#shell").hidden = false;
  $("#userChip").textContent = `${me.username} · ${me.role}`;
  const items = [...menu.map(m => m.id), "profile"];
  $("#nav").innerHTML = items.map(k => `<button data-page="${k}">${ICON[k] || "•"} ${esc(labelOf(k))}</button>`).join("");
  $("#nav").querySelectorAll("button").forEach(b => b.onclick = () => go(b.dataset.page));
  go(items.includes(current) ? current : "copilot");
}
function go(page) {
  current = page; document.body.classList.remove("menu-open");
  $("#nav").querySelectorAll("button").forEach(b => b.classList.toggle("active", b.dataset.page === page));
  $("#pageTitle").textContent = labelOf(page);
  $("#view").innerHTML = '<div class="skeleton"></div>';
  ({copilot: viewCopilot, profile: viewProfile, admin: viewAdmin}[page] || (() => viewTable(page)))();
}

/* ---------- generic table pages (+ role-limited actions, quote form) ---------- */
async function viewTable(page) {
  let d;
  try { d = await api("/api/page/" + page, "GET", null, {quiet: true}); }
  catch (e) {
    $("#view").innerHTML = `<div class="card"><h2>${e.status === 403 ? "403 - Access denied" : "Problem"}</h2><p>${esc(e.message)}</p></div>`;
    return;
  }
  const fmt = {margin_pct: v => v + "%"};
  ["cost_usd", "price", "cost", "rate_usd"].forEach(c => fmt[c] = v => "$" + Number(v).toLocaleString());
  const acts = (ACTIONS[page] || []).filter(a => caps.includes(a.cap));
  let h = `<div class="card"><h2>${esc(labelOf(page))}</h2>`;
  if (acts.length) h += `<div class="actions" style="margin-bottom:.8rem">${acts.map((a, i) => `<button class="btn" data-i="${i}">${esc(a.label)}</button>`).join("")}</div>`;
  h += table(d.columns, d.rows, fmt) + "</div>";
  if (page === "quotes" && caps.includes("quotes.create")) {
    h += `<div class="card"><h2>Create a quote</h2><form id="quoteForm" class="row">
      <label>Origin<input name="origin" required></label><label>Destination<input name="destination" required></label>
      <label>Container<select name="container"><option>20ft</option><option selected>40ft</option><option>40ft HC</option></select></label>
      ${me.role === "Customer" ? "" : '<label>Customer ID<input name="customer_id" placeholder="e.g. C001" required></label>'}
      <button class="btn primary" type="submit">Create</button></form></div>`;
  }
  $("#view").innerHTML = h;
  $("#view").querySelectorAll(".actions .btn").forEach(b => b.onclick = async () => {
    try { const r = await api(acts[Number(b.dataset.i)].path, "POST", {}); toast(r.message); viewTable(page); } catch (e) {}
  });
  const qf = $("#quoteForm");
  if (qf) qf.onsubmit = async e => {
    e.preventDefault();
    try { const r = await api("/api/quotes", "POST", Object.fromEntries(new FormData(qf))); toast(r.message); viewTable(page); } catch (x) {}
  };
}

/* ---------- copilot ---------- */
async function viewCopilot() {
  $("#view").innerHTML = `<div class="card">${caps.includes("agents.run") ? '<div class="actions" style="margin-bottom:.8rem"><button class="btn" id="runAgents">Run agents</button></div>' : ""}
    <div class="chat" id="chat"></div>
    <form class="chat-input" id="askForm"><input id="askBox" placeholder="Ask about carriers, routes, shipments, quotes…" autocomplete="off" required>
    <button class="btn primary" type="submit">Ask</button></form></div>`;
  $("#askForm").onsubmit = e => { e.preventDefault(); const q = $("#askBox").value.trim(); if (q) { $("#askBox").value = ""; ask(q); } };
  const ra = $("#runAgents");
  if (ra) ra.onclick = async () => { try { const r = await api("/api/agents/run", "POST", {}); toast(r.message); } catch (e) {} };
  try {
    const s = await api("/api/copilot/suggestions", "GET", null, {silent: true});
    addBot(`<p>Hi ${esc(me.username)}! Ask me a question — every answer comes from a real database query.</p>${chips(s.suggestions)}`);
  } catch (e) {}
}
const chips = list => `<div class="chips">${list.map(q => `<button type="button" data-q="${esc(q)}">${esc(q)}</button>`).join("")}</div>`;
function addMsg(cls, html) {
  const d = document.createElement("div"); d.className = "msg " + cls; d.innerHTML = html;
  $("#chat").appendChild(d); $("#chat").scrollTop = $("#chat").scrollHeight;
  d.querySelectorAll(".chips button").forEach(b => b.onclick = () => ask(b.dataset.q));
  return d;
}
const addBot = html => addMsg("bot", html);
async function ask(q) {
  addMsg("user", esc(q));
  const wait = addBot('<span class="dots"><span></span><span></span><span></span></span>');
  try {
    const r = await api("/api/copilot", "POST", {question: q}, {silent: true, quiet: true});
    let h = `<div>${esc(r.answer)}</div>`;
    if (r.interpreted) h += `<div class="meta">Understood as: “${esc(r.interpreted)}”</div>`;
    if (r.sql) {
      h += `<div class="meta">${r.row_count} row(s) returned</div>`;
      h += table(r.columns, r.rows);
      h += `<details><summary>Show SQL used</summary><pre class="sql">${esc(r.sql)}</pre></details>`;
    }
    if (r.suggestions && r.suggestions.length) h += chips(r.suggestions);
    wait.remove(); addBot(h);
  } catch (e) { wait.remove(); addBot(`<span style="color:var(--danger)">${esc(e.message)}</span>`); }
}

/* ---------- profile ---------- */
async function viewProfile() {
  const u = me;
  $("#view").innerHTML = `<div class="card"><h2>Your profile</h2>
    <div class="grid">${[["Username", u.username], ["Email", u.email], ["Role", u.role], ["Joined", fmtDate(u.created_at)], ["Last login", fmtDate(u.last_login)]]
      .map(([l, v]) => `<div class="kpi"><div class="l">${l}</div><div style="font-weight:700;word-break:break-all">${esc(v)}</div></div>`).join("")}</div></div>
    <div class="card"><h2>Change username</h2><form id="nameForm" class="row"><label>New username<input name="username" value="${esc(u.username)}" required minlength="3" maxlength="30"></label>
      <button class="btn primary" type="submit">Save</button></form></div>
    <div class="card"><h2>Change password</h2><form id="pwForm" class="row">
      <label>Current<input name="current_password" type="password" autocomplete="current-password" required></label>
      <label>New<input name="new_password" type="password" autocomplete="new-password" required></label>
      <label>Confirm<input name="confirm_password" type="password" autocomplete="new-password" required></label>
      <button class="btn primary" type="submit">Update</button></form></div>`;
  $("#nameForm").onsubmit = async e => { e.preventDefault();
    try { const r = await api("/api/auth/profile", "PUT", Object.fromEntries(new FormData(e.target))); toast(r.message); start(); } catch (x) {} };
  $("#pwForm").onsubmit = async e => { e.preventDefault();
    const d = Object.fromEntries(new FormData(e.target));
    if (d.new_password !== d.confirm_password) { toast("Passwords do not match.", "error"); return; }
    try { const r = await api("/api/auth/password", "POST", d); toast(r.message); e.target.reset(); } catch (x) {} };
}

/* ---------- admin dashboard ---------- */
async function viewAdmin() {
  try {
    const [k, u, a] = await Promise.all([api("/api/admin/kpis", "GET", null, {quiet: true}), api("/api/admin/users", "GET", null, {quiet: true}), api("/api/admin/activity", "GET", null, {quiet: true})]);
    const kpis = [["Total users", k.total, ""], ...Object.entries(k.by_role).map(([r, n]) => [r, n, ""]), ["Locked users", k.locked, "warn"]];
    const roleOpts = sel => u.roles.map(r => `<option ${r === sel ? "selected" : ""}>${esc(r)}</option>`).join("");
    $("#view").innerHTML = `
      <div class="grid">${kpis.map(([l, n, c]) => `<div class="kpi ${c}"><div class="n">${n}</div><div class="l">${esc(l)}</div></div>`).join("")}</div>
      <div class="card"><h2>Add user</h2><form id="addForm" class="row">
        <label>Email<input name="email" type="email" required></label><label>Username<input name="username" required></label>
        <label>Temporary password<input name="password" type="password" required></label>
        <label>Role<select name="role">${roleOpts("Customer")}</select></label><button class="btn primary" type="submit">Add</button></form></div>
      <div class="card"><h2>Users</h2>${table(["email", "username", "role", "status", "created_at", "last_login", "actions"], u.users, {
        role: (v, r) => `<select data-role-id="${r.id}">${roleOpts(v)}</select>`,
        status: v => `<span class="badge ${v === "Active" ? "ok" : "bad"}">${esc(v)}</span>`,
        created_at: v => esc(fmtDate(v)), last_login: v => esc(fmtDate(v)),
        actions: (v, r) => `<div class="actions"><button class="btn small" data-act="${r.locked ? "unlock" : "lock"}" data-id="${r.id}">${r.locked ? "Unlock" : "Lock"}</button>
          <button class="btn small danger" data-act="delete" data-id="${r.id}" data-email="${esc(r.email)}">Delete</button></div>`})}</div>
      <div class="card"><h2>Activity log (latest 50)</h2>${table(["ts", "actor", "action", "detail"], a.activity)}</div>`;
    $("#addForm").onsubmit = async e => { e.preventDefault();
      try { await api("/api/admin/users", "POST", Object.fromEntries(new FormData(e.target))); toast("User added"); viewAdmin(); } catch (x) {} };
    $("#view").querySelectorAll("[data-role-id]").forEach(s => s.onchange = async () => {
      try { await api(`/api/admin/users/${s.dataset.roleId}/role`, "POST", {role: s.value}); toast("Role updated"); } catch (x) {}
      viewAdmin(); });
    $("#view").querySelectorAll("[data-act]").forEach(b => b.onclick = async () => {
      const act = b.dataset.act, id = b.dataset.id;
      if (act === "delete" && !confirm(`Delete ${b.dataset.email}? This cannot be undone.`)) return;
      try {
        await api(act === "delete" ? `/api/admin/users/${id}` : `/api/admin/users/${id}/${act}`, act === "delete" ? "DELETE" : "POST");
        toast("Done");
      } catch (x) {}
      viewAdmin(); });
  } catch (e) { $("#view").innerHTML = `<div class="card"><h2>${e.status === 403 ? "403 - Access denied" : "Problem"}</h2><p>${esc(e.message)}</p></div>`; }
}

start();

Overwriting static/app.js


In [39]:
%%writefile agent_engine.py
"""Six deterministic FreightQuote agents with persisted run/results.

These are local, explainable agents; they do not call paid external AI APIs.
"""
import json
from datetime import datetime, timezone
import db

AGENTS = ("Route AI", "Spot Quotes", "Carriers", "Weather Risk", "Margin", "Customs")

def _save(agent, status, summary, payload):
    db.execute(
        "INSERT INTO agent_results(agent,status,summary,payload,updated_at) VALUES(?,?,?,?,?) "
        "ON CONFLICT(agent) DO UPDATE SET status=excluded.status,summary=excluded.summary,"
        "payload=excluded.payload,updated_at=excluded.updated_at",
        (agent, status, summary, json.dumps(payload, default=str), db.now()))
    return {"agent": agent, "status": status, "summary": summary, "data": payload}

def _route_ai():
    shipments = db.query("SELECT id,origin,destination FROM shipments ORDER BY id")
    routes = db.query("SELECT * FROM routes")
    updated = 0
    for sh in shipments:
        exact = [r for r in routes if r["origin"].lower()==sh["origin"].lower()
                 and r["destination"].lower()==sh["destination"].lower()]
        candidates = exact or [r for r in routes if r["origin"].lower()==sh["origin"].lower()]
        if not candidates:
            candidates = routes
        if not candidates:
            continue
        chosen = min(candidates, key=lambda r: (r["cost_usd"], r["transit_days"]))
        reason = "Exact origin/destination match" if exact else "Best available route estimate"
        db.execute("INSERT INTO route_decisions(shipment_id,via,cost_usd,transit_days,reason,updated_at) "
                   "VALUES(?,?,?,?,?,?) ON CONFLICT(shipment_id) DO UPDATE SET via=excluded.via,"
                   "cost_usd=excluded.cost_usd,transit_days=excluded.transit_days,reason=excluded.reason,"
                   "updated_at=excluded.updated_at",
                   (sh["id"], chosen["via"], chosen["cost_usd"], chosen["transit_days"], reason, db.now()))
        updated += 1
    return _save("Route AI","completed",f"Calculated route decisions for {updated} shipments.",{"shipments_updated":updated})

def _spot_quotes():
    rows = db.query("SELECT port,carrier,rate_usd FROM carrier_rates ORDER BY port,rate_usd")
    best = {}
    for r in rows:
        best.setdefault(r["port"], r)
    data = list(best.values())
    return _save("Spot Quotes","completed",f"Found lowest spot rates for {len(data)} ports.",{"lowest_rates":data})

def _carriers():
    rows = db.query("SELECT id,name,risk_score,on_time_pct FROM carriers")
    ranked = sorted(rows,key=lambda r: (r["risk_score"], -r["on_time_pct"]))
    for rank,r in enumerate(ranked,1):
        db.execute("UPDATE carriers SET rank=? WHERE id=?",(rank,r["id"]))
    return _save("Carriers","completed",f"Refreshed carrier rankings for {len(rows)} carriers.",{"ranked_carriers":[{"name":r["name"],"rank":i+1,"risk_score":r["risk_score"],"on_time_pct":r["on_time_pct"]} for i,r in enumerate(ranked)]})

def _weather_risk():
    storms = db.query("SELECT name,region,severity,wind_kts FROM storms")
    shipments = db.query("SELECT s.id,s.origin,s.destination,COALESCE(rd.via,'') AS via "
                         "FROM shipments s LEFT JOIN route_decisions rd ON rd.shipment_id=s.id")
    changed = 0
    for sh in shipments:
        text = " ".join([sh["origin"], sh["destination"], sh["via"]]).lower()
        hits = [s for s in storms if s["region"].lower() in text]
        high = any(s["severity"].lower()=="high" or s["wind_kts"]>=65 for s in hits)
        risk = "High" if high else ("Medium" if hits else "None")
        db.execute("UPDATE shipments SET weather_risk=? WHERE id=?",(risk,sh["id"]))
        if high:
            changed += 1
    return _save("Weather Risk","completed",f"Refreshed weather risk; {changed} shipments have high risk.",{"storms_checked":len(storms),"shipments_checked":len(shipments),"high_risk_shipments":changed})

def _margin():
    rows = db.query("SELECT id,cost,price FROM quotes")
    for r in rows:
        margin = round(((float(r["price"])-float(r["cost"]))/float(r["price"]))*100,2) if r["price"] else 0
        db.execute("UPDATE quotes SET margin_pct=? WHERE id=?",(margin,r["id"]))
    return _save("Margin","completed",f"Recalculated margins for {len(rows)} quotes.",{"quotes_updated":len(rows)})

def _customs():
    rows = db.query("SELECT id,customs_status,status FROM shipments")
    held = [r for r in rows if r["customs_status"] in ("On Hold","Pending")]
    return _save("Customs","completed",f"Reviewed {len(rows)} shipments; {len(held)} need customs attention.",{"shipments_reviewed":len(rows),"attention_required":[{"shipment_id":r["id"],"customs_status":r["customs_status"]} for r in held]})

def run_all_agents():
    funcs = (_route_ai,_spot_quotes,_carriers,_weather_risk,_margin,_customs)
    results=[]
    for fn in funcs:
        name = fn.__name__
        try:
            result=fn()
            results.append(result)
        except Exception as exc:
            agent = {"_route_ai":"Route AI","_spot_quotes":"Spot Quotes","_carriers":"Carriers","_weather_risk":"Weather Risk","_margin":"Margin","_customs":"Customs"}[name]
            results.append(_save(agent,"error",str(exc),{}))
    db.execute("INSERT INTO agent_runs(started_at,finished_at,status,summary) VALUES(?,?,?,?)",
               (db.now(),db.now(),"completed" if all(x["status"]=="completed" for x in results) else "partial",
                f"{sum(x['status']=='completed' for x in results)}/6 agents completed"))
    return results

def reroute_shipments():
    # Recompute candidate route decisions, then prioritise safe alternatives for high weather risk.
    results = _route_ai()
    rows = db.query("SELECT s.id,s.weather_risk,rd.via,rd.cost_usd,rd.transit_days FROM shipments s "
                    "LEFT JOIN route_decisions rd ON rd.shipment_id=s.id")
    changed = 0
    for r in rows:
        if r["weather_risk"] == "High":
            alternatives = db.query("SELECT via,cost_usd,transit_days FROM routes WHERE LOWER(via) NOT LIKE '%red sea%' "
                                    "ORDER BY cost_usd,transit_days LIMIT 1")
            if alternatives:
                a=alternatives[0]
                db.execute("UPDATE route_decisions SET via=?,cost_usd=?,transit_days=?,reason=?,updated_at=? WHERE shipment_id=?",
                           (a["via"],a["cost_usd"],a["transit_days"],"Weather avoidance: high-risk storm region",db.now(),r["id"]))
                changed += 1
    return {"route_agent":results,"shipments_checked":len(rows),"weather_avoidance_reroutes":changed}

def refresh_weather():
    return _weather_risk()


Overwriting agent_engine.py


In [40]:
%%writefile agent_worker.py
"""Background scheduler. Set AGENT_INTERVAL_SECONDS to tune refresh cadence."""
import os, threading, time
import agent_engine

_started = False
_lock = threading.Lock()

def _loop():
    interval = max(60, int(os.environ.get("AGENT_INTERVAL_SECONDS","300")))
    while True:
        try:
            # A complete cycle includes the Weather Risk agent, refreshing weather-derived risk.
            agent_engine.run_all_agents()
        except Exception:
            pass
        time.sleep(interval)

def start_worker():
    global _started
    with _lock:
        if _started:
            return
        thread=threading.Thread(target=_loop,name="freightquote-agent-worker",daemon=True)
        thread.start()
        _started=True


Overwriting agent_worker.py


In [41]:
%%writefile static/milestone3.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>FreightQuote AI — Milestone 3 Dashboard</title>
<style>
:root{--navy:#102746;--blue:#2166d5;--ink:#10213d;--muted:#62718a;--line:#dce5f3;--bg:#f3f6fc;--green:#07965b;--red:#b91c1c;--amber:#e99a08}
*{box-sizing:border-box}body{margin:0;background:var(--bg);font:15px/1.45 Inter,Arial,sans-serif;color:var(--ink)}
.login{max-width:420px;margin:8vh auto;background:white;padding:28px;border-radius:18px;box-shadow:0 12px 40px #1b37621c}
input{width:100%;padding:12px;margin:7px 0 13px;border:1px solid var(--line);border-radius:9px;font-size:15px}
button{border:0;border-radius:9px;padding:11px 15px;background:var(--blue);color:white;font-weight:700;cursor:pointer}
button.secondary{background:#e8effb;color:#17427d}button.danger{background:#b91c1c}button:disabled{opacity:.5;cursor:not-allowed}
#app{display:none;min-height:100vh}.sidebar{position:fixed;inset:0 auto 0 0;width:215px;background:linear-gradient(180deg,#102746,#0b1b33);color:#fff;padding:25px 15px}
.brand{font-size:21px;font-weight:800;margin:10px 5px 5px}.tag{font-size:12px;color:#c7d7ef;margin:0 5px 35px}.nav{padding:12px 13px;margin:7px 0;border-radius:9px;color:#e5efff}.nav.active{background:#2166d5}
.main{margin-left:215px;padding:22px;max-width:1700px}.top{display:flex;justify-content:space-between;align-items:center;margin-bottom:15px;color:var(--muted)}
.hero{border-radius:16px;padding:22px 25px;background:linear-gradient(110deg,#e8efff,#c8dcff);margin-bottom:15px}
.hero h1{font-size:29px;margin:0 0 4px}.hero p{margin:0;color:#25446f;font-size:17px}
.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:13px;margin:14px 0 18px}.card{background:#fff;border:1px solid var(--line);border-radius:13px;padding:16px;box-shadow:0 3px 12px #1b37620a}
.kpi{display:flex;gap:13px;align-items:center}.bubble{width:55px;height:55px;border-radius:50%;display:grid;place-items:center;font-size:27px;background:#e8f0ff}.num{font-size:29px;font-weight:800}.sub{color:var(--muted)}
.columns{display:grid;grid-template-columns:1.25fr .9fr;gap:16px}.heading{background:var(--navy);color:white;padding:13px 16px;margin:-16px -16px 13px;border-radius:12px 12px 0 0;font-size:18px;font-weight:800}
.agents{display:grid;grid-template-columns:repeat(3,1fr);gap:10px}.agent{border:1px solid var(--line);border-radius:10px;padding:13px;text-align:center;min-height:125px}.agent .ico{font-size:26px}.agent b{display:block;margin:3px}.status{font-size:12px;font-weight:800;color:var(--green)}
.storm{background:#fff7f7;border-color:#f0c7c7}.alert{background:#fee2e2;color:#a31616;padding:10px;border-radius:8px;font-weight:800;margin:10px 0}.success{background:#e6f7ed;color:#087443;padding:11px;border-radius:8px;margin-top:12px}
.metrics{display:grid;grid-template-columns:1fr 1fr;gap:10px}.metric{padding:9px;background:#fff;border:1px solid var(--line);border-radius:8px}
.bottom{display:grid;grid-template-columns:.9fr 1.3fr .8fr;gap:15px;margin-top:16px}.line{padding:8px 0;border-bottom:1px solid #edf1f7;font-size:13px}
.bar{height:9px;background:#e2e9f5;border-radius:9px;overflow:hidden;margin:10px 0}.bar span{display:block;height:100%;width:100%;background:#1672e8}
.actions{display:flex;flex-wrap:wrap;gap:8px;margin:14px 0}.table{width:100%;border-collapse:collapse;font-size:13px}.table td{padding:8px 4px;border-bottom:1px solid #e8edf5}
.note{color:var(--muted);font-size:12px;margin-top:12px}
@media(max-width:1000px){.sidebar{position:static;width:auto;height:auto}.nav{display:inline-block;margin:4px}.main{margin:0}.kpis{grid-template-columns:repeat(2,1fr)}.columns,.bottom{grid-template-columns:1fr}}
@media(max-width:600px){.agents{grid-template-columns:repeat(2,1fr)}.kpis{grid-template-columns:1fr 1fr}.hero h1{font-size:23px}}
</style></head>
<body>
<section class="login" id="loginBox">
<h2>FreightQuote AI</h2><p class="sub">Sign in to open the Milestone 3 dashboard.</p>
<label>Email</label><input id="email" value="admin@freightquote.ai">
<label>Password</label><input id="password" type="password" placeholder="Enter your account password">
<button onclick="login()">Sign in</button><p id="loginMsg" class="note">Demo admin account is created by the notebook seed data. Use the password shown in the notebook's test cell.</p>
</section>
<div id="app">
<aside class="sidebar"><div style="font-size:38px">🚢</div><div class="brand">FreightQuote AI</div><div class="tag">Smarter Logistics. Safer Routes.</div>
<div class="nav active">⌂ &nbsp; Dashboard</div><div class="nav">▶ &nbsp; Run Agents</div><div class="nav">⇄ &nbsp; Re-route</div><div class="nav">☁ &nbsp; Storm Injection</div><div class="nav">☼ &nbsp; Weather Refresh</div><div class="nav">☷ &nbsp; Activity Log</div><div class="nav">▥ &nbsp; Analytics</div>
<div class="note" style="color:#b7f5d1;border:1px solid #3d9d83;padding:12px;border-radius:9px;margin-top:35px">Milestone 3<br>Six AI Agents + Background Worker ✓</div>
</aside>
<main class="main">
<div class="top"><span>Operations Dashboard</span><span>🟢 System Online &nbsp; | &nbsp; <span id="clock"></span></span></div>
<section class="hero"><h1>Welcome to FreightQuote AI</h1><p>Your intelligent logistics assistant</p></section>
<section class="kpis">
<div class="card kpi"><div class="bubble">🚚</div><div><div class="num" id="shipments">—</div><div class="sub">Total Shipments</div></div></div>
<div class="card kpi"><div class="bubble">✅</div><div><div class="num" id="agentCount">—</div><div class="sub">Agents Completed</div></div></div>
<div class="card kpi"><div class="bubble">⚠️</div><div><div class="num" id="highRisk">—</div><div class="sub">High Risk Shipments</div></div></div>
<div class="card kpi"><div class="bubble">💰</div><div><div class="num" id="profit">—</div><div class="sub">Estimated Quote Profit</div></div></div>
</section>
<div class="actions">
<button onclick="action('/api/agents/run','All six agents executed')">▶ Run Agents</button>
<button class="secondary" onclick="action('/api/route/reroute','Route recalculation completed')">⇄ Re-route</button>
<button class="danger" onclick="action('/api/weather/storm','Storm injected; weather refreshed and rerouting attempted')">☁ Inject Storm</button>
<button class="secondary" onclick="action('/api/agents/weather/refresh','Weather refreshed')">↻ Weather Refresh</button>
<button class="secondary" onclick="refresh()">Refresh Dashboard</button>
</div>
<div class="columns">
<section class="card"><div class="heading">⚙ Six AI Agents Execution Results <span style="float:right;color:#8ff0b5">● Live</span></div><div class="agents" id="agents">
<div class="agent">📍<b>Route AI</b><span class="status">Waiting for data</span></div><div class="agent">🏷<b>Spot Quotes</b><span class="status">Waiting for data</span></div><div class="agent">🚚<b>Carriers</b><span class="status">Waiting for data</span></div><div class="agent">☁<b>Weather Risk</b><span class="status">Waiting for data</span></div><div class="agent">📈<b>Margin</b><span class="status">Waiting for data</span></div><div class="agent">🛡<b>Customs</b><span class="status">Waiting for data</span></div></div>
</section>
<section class="card storm"><div class="heading" style="background:#a91f25">☁ Storm Alert &amp; Re-routing</div><div class="alert" id="stormName">Weather risk information loading…</div><div class="metrics"><div class="metric">Weather Risk: <b id="risk">—</b></div><div class="metric">Shipments Checked: <b id="checked">—</b></div><div class="metric">High-risk Loads: <b id="stormHigh">—</b></div><div class="metric">Route Decisions: <b id="routeCount">—</b></div></div><div class="success" id="rerouteMsg">Run Weather Refresh or Inject Storm to update risk and routes.</div></section>
</div>
<div class="bottom">
<section class="card"><div class="heading">◷ Background Worker</div><b>Worker status: Running with API</b><div class="line">Cycle interval: <span id="interval">—</span></div><div class="line">Weather refresh: Included in agent cycle</div><div class="bar"><span></span></div><div class="note">The worker runs a six-agent cycle periodically in the background.</div></section>
<section class="card"><div class="heading">☷ Recent Activity</div><div id="activity">Loading activity…</div></section>
<section class="card"><div class="heading">▤ Database Summary</div><table class="table" id="dbSummary"><tr><td>Agent results</td><td>—</td></tr></table></section>
</div><p class="note" id="message"></p>
</main></div>
<script>
const $=id=>document.getElementById(id);
function setMsg(t){$('message').textContent=t}
async function login(){
 $('loginMsg').textContent='Signing in…';
 try{let r=await fetch('/api/auth/login',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({email:$('email').value,password:$('password').value}),credentials:'same-origin'});let d=await r.json();if(!r.ok)throw Error(d.detail||'Login failed');$('loginBox').style.display='none';$('app').style.display='block';await refresh()}
 catch(e){$('loginMsg').textContent=e.message}
}
async function api(path,opts={}){let r=await fetch(path,{...opts,credentials:'same-origin',headers:{'Content-Type':'application/json',...(opts.headers||{})}});let d=await r.json().catch(()=>({}));if(!r.ok)throw Error(d.detail||d.message||('HTTP '+r.status));return d}
async function action(path,label){try{setMsg('Running: '+label+'…');let d=await api(path,{method:'POST',body:'{}'});setMsg((d.message||label)+' ✓');await refresh()}catch(e){setMsg('Action failed: '+e.message)}}
async function refresh(){
 try{
  const d=await api('/api/dashboard/summary');
  $('shipments').textContent=d.shipments.total||0;$('highRisk').textContent=d.shipments.high_risk||0;
  $('profit').textContent='$'+Number(d.quotes.profit||0).toLocaleString(undefined,{maximumFractionDigits:0});
  const done=d.agents.filter(a=>a.status==='completed').length;$('agentCount').textContent=done+'/6';
  $('agents').innerHTML=d.agents.map(a=>`<div class="agent"><div class="ico">${({'Route AI':'📍','Spot Quotes':'🏷','Carriers':'🚚','Weather Risk':'☁','Margin':'📈','Customs':'🛡'})[a.agent]||'🤖'}</div><b>${a.agent}</b><span class="status">${a.status.toUpperCase()}</span><div class="note">${a.summary}</div></div>`).join('')||'Agents have not run yet.';
  let wr=d.agents.find(a=>a.agent==='Weather Risk');let payload={};try{payload=JSON.parse(wr?.payload||'{}')}catch(e){}
  $('checked').textContent=payload.shipments_checked??'—';$('stormHigh').textContent=payload.high_risk_shipments??d.shipments.high_risk??'—';
  $('risk').textContent=(Number(payload.high_risk_shipments||0)>0?'HIGH':'Normal');
  $('routeCount').textContent=d.routes.total||0;
  $('stormName').textContent=d.latest_storm?`${d.latest_storm.name} — ${d.latest_storm.region} (${d.latest_storm.severity}, ${d.latest_storm.wind_kts} knots)`:'No storm injected in demo data';
  $('rerouteMsg').textContent='Latest Weather Risk result: '+(wr?.summary||'Not run yet')+'. Use Re-route after refresh to recalculate route decisions.';
  $('interval').textContent=Math.max(1,Math.round(d.worker_interval_seconds/60))+' minutes';
  $('activity').innerHTML=d.activity.map(a=>`<div class="line">🟢 ${a.ts||''} — <b>${a.action}</b><br><span class="sub">${a.detail||''}</span></div>`).join('')||'No activity recorded yet.';
  $('dbSummary').innerHTML=`<tr><td>agent_results</td><td>${d.agents.length}</td></tr><tr><td>agent_runs</td><td>${d.runs.length} recent</td></tr><tr><td>route_decisions</td><td>${d.routes.total}</td></tr><tr><td>weather-risk shipments</td><td>${d.shipments.high_risk||0}</td></tr><tr><td>quotes</td><td>${d.quotes.total}</td></tr>`;
 }catch(e){setMsg('Dashboard refresh failed: '+e.message);if(String(e.message).includes('401')){$('app').style.display='none';$('loginBox').style.display='block'}}
}
$('clock').textContent=new Date().toLocaleString();setInterval(()=>{if($('app').style.display==='block')refresh()},30000);
</script></body></html>


Overwriting static/milestone3.html


In [42]:
import sys, os, time, threading, requests, uvicorn, subprocess
sys.path.insert(0, os.getcwd())
try:                                   # allow re-running this cell
    server.should_exit = True
    time.sleep(1.5)
except NameError:
    pass
for m in ["db", "security", "mailer", "auth", "rbac", "analyst", "intent_router", "qa", "main"]:
    sys.modules.pop(m, None)
import main

server = uvicorn.Server(uvicorn.Config(main.app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(40):
    try:
        if requests.get("http://127.0.0.1:8000/api/health", timeout=1).ok:
            break
    except Exception:
        time.sleep(0.25)

if os.environ.get("NGROK_AUTHTOKEN"):
    # Forcibly terminate any system ngrok processes that might be holding the socket
    try:
        subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
        time.sleep(1.5)
    except Exception:
        pass

    from pyngrok import ngrok, process
    try:
        for tunnel in ngrok.get_tunnels():
            ngrok.disconnect(tunnel.public_url)
    except Exception:
        pass
    ngrok.kill()

    try:
        ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
    except Exception as e:
        if "already set" not in str(e).lower():
            raise e

    try:
        # Attempt connection using standard subdomain syntax
        url = ngrok.connect(8000, subdomain="dice-tank-porcupine").public_url
    except Exception as err:
        print(f"[ngrok fallback] Could not bind custom domain, launching dynamic tunnel. Error: {err}")
        try:
            url = ngrok.connect(8000).public_url
        except Exception:
            from google.colab.output import eval_js
            url = eval_js("google.colab.kernel.proxyPort(8000)")
else:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(8000)")
from IPython.display import HTML, display
display(HTML(f'<h3>✅ App is running → <a href="{url}" target="_blank">{url}</a></h3>'))

[ngrok fallback] Could not bind custom domain, launching dynamic tunnel. Error: ngrok client exception, API returned 400: {"error_code":103,"status_code":400,"msg":"invalid endpoint configuration","details":{"err":"yaml: unmarshal errors:\n  line 1: field subdomain not found in type config.Endpoint"}}



In [43]:
import requests, sqlite3, db
B = "http://127.0.0.1:8000"
ACCOUNTS = {"Admin": ("admin@freightquote.ai", "Admin@123"), "Operations Manager": ("ops@freightquote.ai", "Ops@12345"),
            "Freight Broker": ("broker@freightquote.ai", "Broker@123"), "Auditor": ("auditor@freightquote.ai", "Audit@1234"),
            "Customer": ("customer@freightquote.ai", "Customer@123")}
ROLES = list(ACCOUNTS)

def login(e, p):
    s = requests.Session()
    r = s.post(B + "/api/auth/login", json={"email": e, "password": p})
    assert r.status_code == 200, (e, r.status_code, r.text)
    return s
S = {r: login(*c) for r, c in ACCOUNTS.items()}
def check(name, ok): print("✅" if ok else "❌", name)
def get(role, path): return S[role].get(B + path)

TEMP = ["temp.user@freightquote.ai", "lock.me@freightquote.ai", "iso.a@freightquote.ai", "iso.b@freightquote.ai"]
def cleanup():
    with sqlite3.connect(db.DB_PATH) as c:
        for e in TEMP:
            c.execute("DELETE FROM users WHERE email=?", (e,))
        c.execute("DELETE FROM quotes WHERE origin LIKE 'TEST-%'")
cleanup()
A = S["Admin"]
def add(email, uname, pw, role): return A.post(B + "/api/admin/users", json={"email": email, "username": uname, "password": pw, "role": role})
def users(): return A.get(B + "/api/admin/users").json()["users"]

print("--- Admin-only access")
check("Admin opens admin API", get("Admin", "/api/admin/users").status_code == 200)
for r in ROLES[1:]:
    check(f"{r} gets 403 on /api/admin/users", get(r, "/api/admin/users").status_code == 403)
check("No login -> 401", requests.get(B + "/api/page/route").status_code == 401)

print("--- Page permissions (server-side)")
expect = {"route": ["Admin", "Operations Manager", "Auditor"], "quotes": ["Admin", "Freight Broker", "Auditor", "Customer"],
          "carriers": ["Admin", "Operations Manager", "Freight Broker", "Auditor"], "weather": ["Admin", "Operations Manager", "Auditor"],
          "margin": ["Admin", "Freight Broker", "Auditor"], "customs": ["Admin", "Operations Manager", "Auditor"]}
for page, allowed in expect.items():
    ok = all((get(r, f"/api/page/{page}").status_code == 200) == (r in allowed) for r in ROLES)
    check(f"/api/page/{page} allowed only for {', '.join(allowed)}", ok)

print("--- Role-limited actions (Admin + Operations Manager only)")
for name, path in [("re-route", "/api/route/reroute"), ("inject storm", "/api/weather/storm"),
                   ("edit customs", "/api/customs/edit"), ("run agents", "/api/agents/run")]:
    ok = all((S[r].post(B + path, json={}).status_code == 200) == (r in ("Admin", "Operations Manager")) for r in ROLES)
    check(f"{name}: allowed only for Admin + Operations Manager", ok)

print("--- Creating quotes (Admin, Broker, Customer yes; Ops, Auditor no)")
for r in ROLES:
    code_ = S[r].post(B + "/api/quotes", json={"origin": "TEST-A", "destination": "TEST-B", "container": "40ft", "customer_id": "C001"}).status_code
    check(f"{r} create quote -> {code_}", code_ == (201 if r in ("Admin", "Freight Broker", "Customer") else 403))

print("--- Customer isolation")
check("add customer iso.a", add("iso.a@freightquote.ai", "iso_a", "Iso@12345", "Customer").status_code == 200)
check("add customer iso.b", add("iso.b@freightquote.ai", "iso_b", "Iso@12345", "Customer").status_code == 200)
ia, ib = login("iso.a@freightquote.ai", "Iso@12345"), login("iso.b@freightquote.ai", "Iso@12345")
qa_ = ia.post(B + "/api/quotes", json={"origin": "TEST-A", "destination": "TEST-B", "customer_id": "SOMEONE-ELSE"})
check("customer cannot create for another customer", qa_.status_code == 201 and qa_.json()["quote"]["customer_id"] != "SOMEONE-ELSE")
ib.post(B + "/api/quotes", json={"origin": "TEST-C", "destination": "TEST-D"})
ra, rb = ia.get(B + "/api/quotes").json()["quotes"], ib.get(B + "/api/quotes").json()["quotes"]
ca, cb = {q["customer_id"] for q in ra}, {q["customer_id"] for q in rb}
check("each customer sees only their own customer_id", len(ca) == 1 and len(cb) == 1 and ca != cb)
check("customer B cannot open customer A's quote", ib.get(f"{B}/api/quotes/{qa_.json()['quote']['id']}").status_code == 403)
cq = get("Customer", "/api/page/quotes").json()["rows"]
check("demo Customer page shows only C001", len(cq) > 0 and all(r["customer_id"] == "C001" for r in cq))

print("--- Admin actions")
check("Weak password rejected", add("x@y.com", "xuser", "weak", "Auditor").status_code == 400)
check("Bad email rejected", add("nope", "xuser", "Temp@1234", "Auditor").status_code == 400)
check("Add user works", add("temp.user@freightquote.ai", "temp_user", "Temp@1234", "Auditor").status_code == 200)
tmp = next(u for u in users() if u["email"] == "temp.user@freightquote.ai")
admin_id = next(u["id"] for u in users() if u["email"] == "admin@freightquote.ai")
check("Password hash never exposed by API", "password_hash" not in tmp)
check("Promote/demote role", A.post(B + f"/api/admin/users/{tmp['id']}/role", json={"role": "Freight Broker"}).status_code == 200)
ts = login("temp.user@freightquote.ai", "Temp@1234")
check("Token works before lock", ts.get(B + "/api/auth/me").status_code == 200)
A.post(B + f"/api/admin/users/{tmp['id']}/lock")
check("Existing token stops working immediately after lock (423)", ts.get(B + "/api/auth/me").status_code == 423)
check("Locked user cannot log in (423)", requests.post(B + "/api/auth/login", json={"email": "temp.user@freightquote.ai", "password": "Temp@1234"}).status_code == 423)
A.post(B + f"/api/admin/users/{tmp['id']}/unlock")
check("Unlock restores login", requests.post(B + "/api/auth/login", json={"email": "temp.user@freightquote.ai", "password": "Temp@1234"}).status_code == 200)
check("Admin cannot delete self", A.delete(B + f"/api/admin/users/{admin_id}").status_code == 400)
check("Admin cannot lock self", A.post(B + f"/api/admin/users/{admin_id}/lock").status_code == 400)
check("Delete user works", A.delete(B + f"/api/admin/users/{tmp['id']}").status_code == 200)
print("   KPIs:", A.get(B + "/api/admin/kpis").json())
act = A.get(B + "/api/admin/activity").json()["activity"]
check(f"Activity log recorded actions (showing {len(act)}, max 50)", 0 < len(act) <= 50)

print("--- Login security (5 wrong passwords lock the account)")
add("lock.me@freightquote.ai", "lock_me", "Lock@1234", "Customer")
unk = requests.post(B + "/api/auth/login", json={"email": "nobody@nowhere.com", "password": "wrong"}).json()["detail"]
resps = [requests.post(B + "/api/auth/login", json={"email": "lock.me@freightquote.ai", "password": "wrong"}) for _ in range(5)]
check("Same error for unknown email and wrong password", resps[0].json()["detail"] == unk)
check("Locked on 5th wrong attempt [401,401,401,401,423]", [r.status_code for r in resps] == [401, 401, 401, 401, 423])
f1 = requests.post(B + "/api/auth/forgot", json={"email": "lock.me@freightquote.ai"})
f2 = requests.post(B + "/api/auth/forgot", json={"email": "nobody@nowhere.com"})
check("Forgot-password gives the same reply for known and unknown emails", f1.status_code == f2.status_code == 200 and f1.json() == f2.json())

cleanup()

--- Admin-only access
✅ Admin opens admin API
✅ Operations Manager gets 403 on /api/admin/users
✅ Freight Broker gets 403 on /api/admin/users
✅ Auditor gets 403 on /api/admin/users
✅ Customer gets 403 on /api/admin/users
✅ No login -> 401
--- Page permissions (server-side)
✅ /api/page/route allowed only for Admin, Operations Manager, Auditor
✅ /api/page/quotes allowed only for Admin, Freight Broker, Auditor, Customer
✅ /api/page/carriers allowed only for Admin, Operations Manager, Freight Broker, Auditor
✅ /api/page/weather allowed only for Admin, Operations Manager, Auditor
✅ /api/page/margin allowed only for Admin, Freight Broker, Auditor
✅ /api/page/customs allowed only for Admin, Operations Manager, Auditor
--- Role-limited actions (Admin + Operations Manager only)
✅ re-route: allowed only for Admin + Operations Manager
✅ inject storm: allowed only for Admin + Operations Manager
✅ edit customs: allowed only for Admin + Operations Manager
✅ run agents: allowed only for Admin + Opera

In [44]:
QUESTIONS = ["Which carrier is cheapest to Rotterdam?", "How many shipments are on customs hold?", "Show quotes below 10% margin.",
 "Which ports are most congested?", "Show storms near the Red Sea.", "Which carrier has the highest risk?", "Show available shipping routes.",
 "Which route has the lowest cost?", "How many quotes are generated?", "Show shipments with weather risks.", "Which carrier has the best ranking?",
 "Show the delayed shipments.", "How many shipments are in transit?", "Which ports have high congestion?", "Show customs-related shipment information.",
 "Which route is the fastest?", "What is the average margin?", "Show shipment status summary.", "Show all carriers.", "Show my latest quotes.",
 "whch carier is chepest to rotterdm", "how many shpments r on custom hold", "hello", "What is the capital of France?"]
def ask_(role, q): return S[role].post(B + "/api/copilot", json={"question": q}).json()
tested = []
for q in QUESTIONS:
    r = ask_("Admin", q); tested.append((q, r))
    print(f"Q: {q}\n   A: {r['answer']}\n   rows={r['row_count']}  SQL={r['sql']}\n")
print("--- RBAC in the Copilot")
r = ask_("Customer", "How many quotes are generated?"); print("Customer:", r["answer"], "|", r["sql"])
check("Customer Copilot is limited to own quotes", "customer_id = 'C001'" in (r["sql"] or ""))
check("Customer is refused carrier data", ask_("Customer", "Which carrier has the highest risk?").get("denied") is True)
check("Broker is refused shipment/route data", ask_("Freight Broker", "How many shipments are in transit?").get("denied") is True)
check("Out-of-scope question suggests 3 valid questions", len(ask_("Admin", "What is the capital of France?")["suggestions"]) == 3)

Q: Which carrier is cheapest to Rotterdam?
   A: Maersk offers the cheapest rate to Rotterdam: $1,773 per TEU.
   rows=3  SQL=SELECT carrier, port, rate_usd FROM carrier_rates WHERE port = 'Rotterdam' ORDER BY rate_usd ASC LIMIT 3

Q: How many shipments are on customs hold?
   A: 2 shipment(s) are currently on customs hold.
   rows=1  SQL=SELECT COUNT(*) AS shipments_on_customs_hold FROM shipments WHERE customs_status = 'On Hold'

Q: Show quotes below 10% margin.
   A: 3 quote(s) have a margin below 10%.
   rows=3  SQL=SELECT id, customer_id, origin, destination, carrier, margin_pct FROM quotes WHERE margin_pct < 10 ORDER BY margin_pct ASC

Q: Which ports are most congested?
   A: Shanghai is the most congested port (congestion index 91).
   rows=5  SQL=SELECT name AS port, country, congestion_index, congestion_level FROM ports ORDER BY congestion_index DESC LIMIT 5

Q: Show storms near the Red Sea.
   A: 5 storm(s) found near Red Sea. Strongest: Storm Injected-5 (High, 80 kts).
   row

In [45]:
import rbac
pages = ["copilot", "route", "quotes", "carriers", "weather", "margin", "customs", "admin"]
roles = rbac.ROLES
mx = "| Page / Feature | " + " | ".join(roles) + " |\n|---|" + "---|" * len(roles) + "\n"
for p in pages:
    mx += f"| {p.title()} | " + " | ".join("Yes" if p in rbac.ROLE_MENU[r] else "No" for r in roles) + " |\n"
for label, cap in [("Run agents", "agents.run"), ("Inject storms", "weather.inject_storm"), ("Edit customs", "customs.edit"),
                   ("Re-route", "route.reroute"), ("Create quotes", "quotes.create")]:
    mx += f"| {label} | " + " | ".join("Yes" if cap in rbac.ROLE_CAPS[r] else "No" for r in roles) + " |\n"
qa_rows = "\n".join(f"| {q} | {r['answer']} | `{r['sql'] or '-'}` | {r['row_count']} |" for q, r in tested)
readme = f"""# FreightQuote AI - Milestone 3 (Six Agents + Background Worker + Weather Re-routing)

## How to run
1. Open the notebook in Google Colab (optionally add Colab Secrets `JWT_SECRET`, `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD`).
2. Runtime -> Run all. Open the link printed by the Launch cell.
3. Demo logins are listed at the top of the notebook.

## Roles and privileges
{mx}
## Features
- Login / signup / forgot + reset password / profile (bcrypt hashing, JWT, lock after 5 wrong passwords)
- Server-side RBAC (HTTP 403) on every route; customers only see their own quotes
- Admin dashboard: KPI cards, users table, add / delete / promote / demote / lock / unlock, activity log
- AI Copilot: 20 SQL intents, spelling tolerance, shows SQL + row count, RBAC-aware
- Six deterministic agents: Route AI, Spot Quotes, Carriers, Weather Risk, Margin, Customs
- Background agent worker (default 5-minute cycle; configure AGENT_INTERVAL_SECONDS)
- Persisted agent results, run history, and shipment route decisions
- Storm injection triggers weather-risk refresh and weather-aware route recalculation
- UI: dark mode, responsive layout, loading and error states

## Tested Copilot questions (answers come from live SQL)
| Question | Answer | SQL | Rows |
|---|---|---|---|
{qa_rows}

## Before pushing to GitHub
Clear notebook outputs and delete `freightquote_full.db`. Never commit secrets or tokens.
"""
open("README.md", "w").write(readme)
print(readme[:1500], "...")

# FreightQuote AI - Milestone 3 (Six Agents + Background Worker + Weather Re-routing)

## How to run
1. Open the notebook in Google Colab (optionally add Colab Secrets `JWT_SECRET`, `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD`).
2. Runtime -> Run all. Open the link printed by the Launch cell.
3. Demo logins are listed at the top of the notebook.

## Roles and privileges
| Page / Feature | Admin | Operations Manager | Freight Broker | Auditor | Customer |
|---|---|---|---|---|---|
| Copilot | Yes | Yes | Yes | Yes | Yes |
| Route | Yes | Yes | No | Yes | No |
| Quotes | Yes | No | Yes | Yes | Yes |
| Carriers | Yes | Yes | Yes | Yes | No |
| Weather | Yes | Yes | No | Yes | No |
| Margin | Yes | No | Yes | Yes | No |
| Customs | Yes | Yes | No | Yes | No |
| Admin | Yes | No | No | No | No |
| Run agents | Yes | Yes | No | No | No |
| Inject storms | Yes | Yes | No | No | No |
| Edit customs | Yes | Yes | No | No | No |
| Re-route | Yes | Yes | No | No | No |
| Create quotes | Y